# DCC 2026 Mission 2 — MFCC40 Feature Experiments

이 Notebook은 검증된 **MFCC40 + StandardScaler + Logistic Regression**을 고정 비교점으로 사용한다. E00~E02 실행·검토 이후, E03=`zcr`와 E04=`spectral_centroid`는 official Validation 결과를 보기 전에 기존 Training-only Stage 2 Accuracy 순위로 사전 확정했다.

공용 `mission2_features.py` / `mission2_train.py`의 53차원 GBM 계약은 이 Notebook에 섞지 않는다. 모든 대규모 실행 switch는 기본 `False`이고, cache가 없다고 자동 추출하지 않는다. 저장된 E00/E01 출력과 수치는 변경하지 않으며, 디스크에 없는 E02 report/table row를 하드코딩하지 않는다.

### Candidate cache v1/v2 provenance (code-only review)

기존 E02/E05 저장 수치와 출력은 **historical v1 candidate cache 결과**다. 아래의 현재 v2 추출 코드로 재실행해 얻은 수치가 아니다.

v2는 512 sample 미만 발화에서 frame energy와 ZCR을 실제 발화 sample만으로 정의한다. E02의 `spectral_rolloff` 계산 경로는 유지되지만, E05의 `zcr` 값은 짧은 발화가 있다면 달라질 수 있다. 해당 발화의 실제 개수와 metric 영향은 이번 코드 수정에서 확인하지 않았다.

현재 v2 loader는 v1 cache metadata를 거부한다. v2 결과를 주장하려면 기존 cache를 보존한 별도 경로에 v2 candidate cache를 생성하고, 해당 실험을 다시 실행·검증해야 한다. 이번 수정에서는 cache 재생성이나 모델 재평가를 하지 않았다.

## 실행 순서와 중단 원칙

1. Setup → Checkpoint A metadata preflight
2. `RUN_BASELINE_REPRODUCTION=True` → Checkpoint A baseline exact gate
3. 기존 E00/E01/E02 기록과 E03/E04 Training-only precommit 확인
4. E03 설정 하나만 입력 → selection lock → Training cache 검증/재사용 → E03 Validation cache 검증 또는 명시적 생성
5. `RUN_SELECTED_FEATURE_EXPERIMENT=True`로 E03만 실행 → Checkpoint G → E03 해석 기록
6. E03 결과와 무관하게 E04 설정 하나만 입력 → selection lock → cache 검증/재사용 → E04만 실행
7. E04 실행 뒤 Checkpoint G와 해석 기록

assertion 또는 `RuntimeError`가 발생하면 후속 cell을 건너뛰지 말고 즉시 중단한다.
향후 v2 재실행은 Training-only Stage 2를 실제로 실행하고(E05는 두 feature 조합 포함), metric/split/data 근거를 동결한 뒤 Official Validation을 연다. 과거 precommit 기록은 v2 동결 기록이 아니다.


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
if (current / 'src').is_dir():
    REPO_ROOT = current
elif (current.parent / 'src').is_dir():
    REPO_ROOT = current.parent
else:
    raise FileNotFoundError(f'Repository root를 찾지 못했습니다: {current}')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from dcc_data.mission2_candidate_features import (
    CANDIDATE_REGISTRY,
    CANDIDATE_EXTRACTION_CONFIG,
    freeze_stage2_precommit,
    require_stage2_precommit,
    build_candidate_feature_cache,
    candidate_cache_paths,
    candidate_registry_table,
    load_candidate_feature_cache,
    lock_candidate_selection,
    stage1_feasibility_diagnostics,
    stage2_training_group_screen,
    validate_raw_eda_acoustic_metadata,
)
from dcc_data.mission2_experiments import (
    ANNOTATION_TIME_UNIT,
    BASELINE_FEATURE_NAMES,
    EXPECTED_BASELINE_CODE_SHA256,
    EXPECTED_CALL_COUNTS,
    EXPECTED_CACHE_METADATA,
    EXPECTED_UTTERANCE_COUNTS,
    MFCC_CONFIG,
    REFERENCE_ACCURACY,
    REFERENCE_CONFUSION_MATRIX,
    REFERENCE_MODEL_CONFIG,
    SPEAKER_LABEL_MAPPING,
    align_frozen_cache,
    append_comparison_row,
    assemble_experiment_matrix,
    assert_single_factor_comparison,
    build_duration_feature_frame,
    build_experiment_report,
    empty_experiment_table,
    frozen_cache_paths,
    load_frozen_mfcc_cache,
    notebook_code_sha256,
    print_experiment_report,
    run_baseline_reproduction,
    run_feature_experiment,
    validate_cache_metadata,
)

In [2]:
# 경로 정의만 수행한다. 이 cell은 data/cache/model을 읽지 않는다.
DATA_ROOT = Path(os.environ.get('DCC_DATA_ROOT', REPO_ROOT / 'data' / 'raw_original')).expanduser().resolve()
BASELINE_NOTEBOOK_PATH = REPO_ROOT / 'notebooks' / 'mission2_yeonwoo_baseline.ipynb'
BASELINE_CACHE_ROOT = REPO_ROOT / 'data' / 'processed' / 'mission2'
EDA_CACHE_ROOT = REPO_ROOT / 'tmp' / 'mission2_eda'
EDA_MANIFEST_PATH = EDA_CACHE_ROOT / 'utterance_manifest_v1.parquet'
EDA_MANIFEST_META_PATH = EDA_CACHE_ROOT / 'utterance_manifest_meta_v1.json'
EDA_ACOUSTIC_PATH = EDA_CACHE_ROOT / 'acoustic_features_sample_v1.parquet'
EDA_ACOUSTIC_META_PATH = EDA_CACHE_ROOT / 'acoustic_features_meta_v1.json'
CANDIDATE_CACHE_ROOT = REPO_ROOT / 'data' / 'processed' / 'mission2_candidate_features'
DECISION_TABLE_PATH = REPO_ROOT / 'reports' / 'mission2' / 'feature_experiment_table.csv'

# 모든 대규모/모델 실행은 사용자가 해당 단계에서 직접 True로 바꾼다.
RUN_BASELINE_REPRODUCTION = False
RUN_E01 = False
RUN_STAGE1_FEASIBILITY = False
BUILD_FULL_TRAINING_CANDIDATE_CACHE = False
RUN_STAGE2_TRAINING_SCREEN = False
LOCK_STAGE2_SELECTION = False
CANDIDATE_SELECTION_LOCKED = False
BUILD_VALIDATION_CANDIDATE_CACHE = False
RUN_SELECTED_FEATURE_EXPERIMENT = False
SAVE_DECISION_TABLE = False

# Stage 1은 목록의 계산 가능성만 진단한다. 순위나 E02 번호를 만들지 않는다.
STAGE1_CANDIDATE_NAMES = tuple(CANDIDATE_REGISTRY)

# Stage 1을 본 뒤 사용자가 채운다. 한 번의 audio pass로 여러 후보를 cache할 수 있다.
FULL_TRAINING_CACHE_FEATURES: tuple[str, ...] = (
    'zcr',
    'spectral_centroid',
    'spectral_rolloff',
    'spectral_flatness',
    'silence_ratio',
)
CANDIDATE_GROUPS: dict[str, tuple[str, ...]] = {}

# E03/E04는 E02 official Validation 결과를 보기 전에 Training-only Stage 2 순위로 사전 확정됐다.
PRECOMMITTED_EXPERIMENTS: dict[str, tuple[str, ...]] = {
    'E03': ('zcr',),
    'E04': ('spectral_centroid',),
}
# 과거 v1 요약이며, 향후 v2의 frozen precommit 근거로 재사용하지 않는다.
TRAINING_ONLY_STAGE2_PRECOMMIT = {
    'source_split': 'Training',
    'source_rows': 889_358,
    'seed': 42,
    'test_size': 0.2,
    'internal_train_rows': 711_467,
    'internal_validation_rows': 177_891,
    'train_calls': 23_360,
    'internal_validation_calls': 5_840,
    'call_id_overlap': 0,
    'official_validation_used': False,
    'accuracy_delta_vs_mfcc40': {
        'spectral_rolloff': 0.001979,
        'zcr': 0.001180,
        'spectral_centroid': 0.001152,
        'spectral_flatness': 0.000416,
        'silence_ratio': -0.000208,
    },
}
DEFERRED_OFFICIAL_VALIDATION_CANDIDATES = ('spectral_flatness', 'silence_ratio')

# 아래 세 값에는 한 번에 E03 또는 E04 하나만 입력한다. 기본값은 실행하지 않는 빈 선택이다.
SELECTED_CANDIDATE_NAME = ''
SELECTED_FEATURE_NAMES: tuple[str, ...] = ()
SELECTED_EXPERIMENT_ID = ''

print('REPO_ROOT:', REPO_ROOT)
print('All execution switches are False. No data/model/cache was read.')

REPO_ROOT: C:\Users\YW\Study\02_Projects\DCC_2026\prelim
All execution switches are False. No data/model/cache was read.


# Checkpoint A — Preflight / Baseline Contract

1. **무엇:** MFCC40 reference의 source, cache, row/key, scaler/model, exact metric 계약을 검사한다.
2. **입력:** baseline Notebook code hash, 작은 JSON metadata, 이후 명시적으로 연 `.npz`와 Raw EDA manifest.
3. **출력 의미:** exact confusion matrix까지 같아야 후속 feature 실험 gate가 열린다.
4. **믿으면 안 되는 경우:** source/cache metadata, row alignment, label, 시간 단위, scaler fit scope 중 하나라도 다르면 중단한다.
5. **다음 판단:** `BASELINE_GATE_PASSED=True`가 생성된 경우에만 E01로 이동한다.

In [3]:
# Small-file/static preflight only: no array load, feature extraction, or fitting.
assert ANNOTATION_TIME_UNIT == 'milliseconds'
assert BASELINE_NOTEBOOK_PATH.is_file(), BASELINE_NOTEBOOK_PATH
actual_baseline_hash = notebook_code_sha256(BASELINE_NOTEBOOK_PATH)
assert actual_baseline_hash == EXPECTED_BASELINE_CODE_SHA256, (
    'Baseline code changed; redefine the reference before experimenting',
    actual_baseline_hash,
    EXPECTED_BASELINE_CODE_SHA256,
)

preflight_rows = []
for split in ['Training', 'Validation']:
    npz_path, metadata_path = frozen_cache_paths(BASELINE_CACHE_ROOT, split)
    assert npz_path.is_file() and metadata_path.is_file(), (npz_path, metadata_path)
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    validate_cache_metadata(metadata, split)
    preflight_rows.append({'split': split, **metadata, 'npz_path': str(npz_path)})
assert EDA_MANIFEST_META_PATH.is_file(), EDA_MANIFEST_META_PATH
eda_manifest_meta = json.loads(EDA_MANIFEST_META_PATH.read_text(encoding='utf-8'))
assert eda_manifest_meta['schema_version'] == 'mission2_raw_eda_v1'
assert eda_manifest_meta['allowed_annotation_fields'] == ['id', 'startAt', 'endAt', 'speaker']
assert eda_manifest_meta['call_id_definition'] == 'file stem; use (split, call_id) as group key'
assert isinstance(eda_manifest_meta['source_signature'], str) and eda_manifest_meta['source_signature']
assert MFCC_CONFIG == {'sample_rate': 8000, 'n_mfcc': 20, 'n_fft': 512, 'hop_length': 160}
assert REFERENCE_MODEL_CONFIG == {
    'solver': 'lbfgs', 'max_iter': 1000, 'class_weight': None, 'random_state': 42
}
assert len(BASELINE_FEATURE_NAMES) == 40
assert SPEAKER_LABEL_MAPPING == {0: '119_dispatcher', 1: 'caller'}
assert REFERENCE_CONFUSION_MATRIX.tolist() == [[41190, 12655], [8715, 49387]]
assert np.isclose(REFERENCE_ACCURACY, 0.809106094848455)
display(pd.DataFrame(preflight_rows))
print('Checkpoint A metadata preflight ready: annotation startAt/endAt unit = milliseconds.')
print('Label mapping:', SPEAKER_LABEL_MAPPING)

,split,cache_schema_version,feature_dimension,mfcc_config,source_call_count,source_split,utterance_count,npz_path
0,Training,mission2_mfcc_mean_std_v1,40,"{'hop_length': 160, 'n_fft': 512, 'n_mfcc': 20...",29200,Training,889358,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...
1,Validation,mission2_mfcc_mean_std_v1,40,"{'hop_length': 160, 'n_fft': 512, 'n_mfcc': 20...",3640,Validation,111947,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...


Checkpoint A metadata preflight ready: annotation startAt/endAt unit = milliseconds.
Label mapping: {0: '119_dispatcher', 1: 'caller'}


In [4]:
if not RUN_BASELINE_REPRODUCTION:
    raise RuntimeError('STOP: metadata preflight를 검토한 뒤 RUN_BASELINE_REPRODUCTION=True로 바꾸세요.')
assert EDA_MANIFEST_PATH.is_file(), EDA_MANIFEST_PATH
full_eda_manifest = pd.read_parquet(EDA_MANIFEST_PATH)
assert len(full_eda_manifest) == sum(EXPECTED_UTTERANCE_COUNTS.values())
assert full_eda_manifest['record_key'].is_unique
assert full_eda_manifest.groupby('split')['call_id'].nunique().to_dict() == EXPECTED_CALL_COUNTS

training_bundle = load_frozen_mfcc_cache(BASELINE_CACHE_ROOT, 'Training')
validation_bundle = load_frozen_mfcc_cache(BASELINE_CACHE_ROOT, 'Validation')
aligned_training = align_frozen_cache(training_bundle, full_eda_manifest, 'Training')
aligned_validation = align_frozen_cache(validation_bundle, full_eda_manifest, 'Validation')
del training_bundle, validation_bundle
assert aligned_training.X_mfcc.shape == (889358, 40)
assert aligned_validation.X_mfcc.shape == (111947, 40)
assert all(str(key).startswith('Training|') for key in aligned_training.record_keys)
assert all(str(key).startswith('Validation|') for key in aligned_validation.record_keys)

baseline_result = run_baseline_reproduction(aligned_training, aligned_validation)
baseline_report = build_experiment_report(
    baseline_result,
    training_tracking=aligned_training.tracking,
    evaluation_tracking=aligned_validation.tracking,
    baseline_report=None,
)
print_experiment_report(baseline_report)
display(baseline_report['duration_summary'])
display(baseline_report['overlap_summary'])
display(baseline_report['confidence_summary'])
BASELINE_GATE_PASSED = True
print('CHECKPOINT A PASS: exact baseline reproduction gate opened.')

Experiment ID: M2-REFERENCE-MFCC40-LR
Feature names: ['mfcc_01_mean', 'mfcc_02_mean', 'mfcc_03_mean', 'mfcc_04_mean', 'mfcc_05_mean', 'mfcc_06_mean', 'mfcc_07_mean', 'mfcc_08_mean', 'mfcc_09_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'mfcc_14_mean', 'mfcc_15_mean', 'mfcc_16_mean', 'mfcc_17_mean', 'mfcc_18_mean', 'mfcc_19_mean', 'mfcc_20_mean', 'mfcc_01_std', 'mfcc_02_std', 'mfcc_03_std', 'mfcc_04_std', 'mfcc_05_std', 'mfcc_06_std', 'mfcc_07_std', 'mfcc_08_std', 'mfcc_09_std', 'mfcc_10_std', 'mfcc_11_std', 'mfcc_12_std', 'mfcc_13_std', 'mfcc_14_std', 'mfcc_15_std', 'mfcc_16_std', 'mfcc_17_std', 'mfcc_18_std', 'mfcc_19_std', 'mfcc_20_std']
Feature dimension: 40
Training shape: (889358, 40)
Validation shape: (111947, 40)
Accuracy: 0.8091060948484551
Confusion matrix: [[41190, 12655], [8715, 49387]]
speaker=0 Recall: 0.7649735351471817
speaker=1 Recall: 0.8500051633334481
Accuracy delta vs baseline: 0.0
Recall 0 delta vs baseline: 0.0
Recall 1 delta vs baseline:

,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,very_short,5786,0.790010,"[[2313, 565], [650, 2258]]",0.803683,0.776479
1,short,22545,0.755068,"[[6978, 2746], [2776, 10045]]",0.717606,0.783480
2,medium,55659,0.797787,"[[22389, 7451], [3804, 22015]]",0.750302,0.852667
3,long,22384,0.871024,"[[8059, 1661], [1226, 11438]]",0.829115,0.903190
4,very_long,5573,0.911897,"[[1451, 232], [259, 3631]]",0.862151,0.933419


,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,no_overlap,63084,0.842940,"[[23996, 5890], [4018, 29180]]",0.802918,0.878969
1,overlap,48863,0.765426,"[[17194, 6765], [4697, 20207]]",0.717643,0.811396


,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,0.50-0.60,14835,0.570745,"[[3088, 3932], [2436, 5379]]",0.439886,0.688292
1,0.60-0.70,20153,0.650077,"[[4084, 5041], [2011, 9017]]",0.447562,0.817646
2,0.70-0.80,14596,0.774322,"[[5929, 1603], [1691, 5373]]",0.787175,0.760617
3,0.80-0.90,20736,0.876061,"[[9715, 1163], [1407, 8451]]",0.893087,0.857273
4,0.90-0.95,15901,0.926546,"[[7570, 503], [665, 7163]]",0.937694,0.915049
5,0.95-0.99,19344,0.958902,"[[8419, 350], [445, 10130]]",0.960087,0.957920
6,0.99-1.00,6382,0.980727,"[[2385, 63], [60, 3874]]",0.974265,0.984748


CHECKPOINT A PASS: exact baseline reproduction gate opened.


# Checkpoint B — E01 Feature Construction

1. **무엇:** 원본 annotation의 ms 계약에서 `duration_ms = endAt_ms - startAt_ms` 한 열을 만든다.
2. **입력:** gate를 통과한 Training/Validation tracking과 동일 MFCC40 행.
3. **출력 의미:** 두 matrix는 각각 41차원이며 앞 40열은 reference와 bitwise 동일하다.
4. **믿으면 안 되는 경우:** key/label/행 순서가 다르거나 absolute timestamp가 feature name에 들어가면 중단한다.
5. **다음 판단:** feature construction guard가 통과한 경우에만 E01 fitting을 허용한다.

In [5]:
assert globals().get('BASELINE_GATE_PASSED', False), 'Checkpoint A gate did not pass'
e01_training_duration = build_duration_feature_frame(aligned_training.tracking)
e01_validation_duration = build_duration_feature_frame(aligned_validation.tracking)
X_e01_training, E01_FEATURE_NAMES = assemble_experiment_matrix(
    aligned_training, e01_training_duration, ['duration_ms']
)
X_e01_validation, E01_VALIDATION_FEATURE_NAMES = assemble_experiment_matrix(
    aligned_validation, e01_validation_duration, ['duration_ms']
)
assert E01_FEATURE_NAMES == E01_VALIDATION_FEATURE_NAMES
assert X_e01_training.shape == (889358, 41)
assert X_e01_validation.shape == (111947, 41)
assert 'startAt_ms' not in E01_FEATURE_NAMES and 'endAt_ms' not in E01_FEATURE_NAMES
assert_single_factor_comparison(
    aligned_training.X_mfcc, X_e01_training, aligned_training.y, aligned_training.y,
    BASELINE_FEATURE_NAMES, E01_FEATURE_NAMES, ['duration_ms']
)
assert_single_factor_comparison(
    aligned_validation.X_mfcc, X_e01_validation, aligned_validation.y, aligned_validation.y,
    BASELINE_FEATURE_NAMES, E01_FEATURE_NAMES, ['duration_ms']
)
E01_FEATURE_CONSTRUCTION_PASSED = True
print('E01 feature shapes:', X_e01_training.shape, X_e01_validation.shape)
print('Only added feature:', E01_FEATURE_NAMES[-1])
del X_e01_training, X_e01_validation

E01 feature shapes: (889358, 41) (111947, 41)
Only added feature: duration_ms


E01 feature 생성 완료

MFCC40 → MFCC40 + duration_ms

입력 차원: 40 → 41

아직 성능 평가는 수행하지 않음

# Checkpoint C — E01 Fair Comparison

1. **무엇:** MFCC40 baseline과 E01을 duration 한 열만 다른 조건에서 비교한다.
2. **입력:** 같은 행/key/label, 같은 LR 설정, 각 실험의 Training-only StandardScaler.
3. **출력 의미:** Accuracy가 primary metric이며 Recall/high-confidence/slice는 보조 진단이다.
4. **믿으면 안 되는 경우:** baseline exact gate, 41차원 guard, fit-scope audit 중 하나라도 실패하면 결과를 사용하지 않는다.
5. **다음 판단:** E01 효과는 실제 출력 후 판단하며 slice 개선만으로 성공이라 부르지 않는다.

In [6]:
assert globals().get('E01_FEATURE_CONSTRUCTION_PASSED', False)
if not RUN_E01:
    raise RuntimeError('STOP: Checkpoint B를 검토한 뒤 RUN_E01=True로 바꾸세요.')
e01_result = run_feature_experiment(
    experiment_id='E01',
    training=aligned_training,
    validation=aligned_validation,
    extra_training=e01_training_duration,
    extra_validation=e01_validation_duration,
    extra_feature_names=['duration_ms'],
    baseline_result=baseline_result,
)
e01_report = build_experiment_report(
    e01_result,
    training_tracking=aligned_training.tracking,
    evaluation_tracking=aligned_validation.tracking,
    baseline_report=baseline_report,
)
print_experiment_report(e01_report)
print('Overlap is evaluation-only metadata and is not in feature_names.')
display(e01_report['duration_summary'])
display(e01_report['overlap_summary'])
display(e01_report['confidence_summary'])

Experiment ID: E01
Feature names: ['mfcc_01_mean', 'mfcc_02_mean', 'mfcc_03_mean', 'mfcc_04_mean', 'mfcc_05_mean', 'mfcc_06_mean', 'mfcc_07_mean', 'mfcc_08_mean', 'mfcc_09_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'mfcc_14_mean', 'mfcc_15_mean', 'mfcc_16_mean', 'mfcc_17_mean', 'mfcc_18_mean', 'mfcc_19_mean', 'mfcc_20_mean', 'mfcc_01_std', 'mfcc_02_std', 'mfcc_03_std', 'mfcc_04_std', 'mfcc_05_std', 'mfcc_06_std', 'mfcc_07_std', 'mfcc_08_std', 'mfcc_09_std', 'mfcc_10_std', 'mfcc_11_std', 'mfcc_12_std', 'mfcc_13_std', 'mfcc_14_std', 'mfcc_15_std', 'mfcc_16_std', 'mfcc_17_std', 'mfcc_18_std', 'mfcc_19_std', 'mfcc_20_std', 'duration_ms']
Feature dimension: 41
Training shape: (889358, 41)
Validation shape: (111947, 41)
Accuracy: 0.8091150276470115
Confusion matrix: [[41225, 12620], [8749, 49353]]
speaker=0 Recall: 0.7656235490760517
speaker=1 Recall: 0.8494199855426664
Accuracy delta vs baseline: 8.93279855640472e-06
Recall 0 delta vs baseline: 0.0006500139288699

,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,very_short,5786,0.787764,"[[2315, 563], [665, 2243]]",0.804378,0.771320
1,short,22545,0.753072,"[[7020, 2704], [2863, 9958]]",0.721925,0.776694
2,medium,55659,0.798308,"[[22519, 7321], [3905, 21914]]",0.754658,0.848755
3,long,22384,0.871337,"[[7964, 1756], [1124, 11540]]",0.819342,0.911244
4,very_long,5573,0.916024,"[[1407, 276], [192, 3698]]",0.836007,0.950643


,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,no_overlap,63084,0.843257,"[[24041, 5845], [4043, 29155]]",0.804423,0.878216
1,overlap,48863,0.765037,"[[17184, 6775], [4706, 20198]]",0.717225,0.811034


,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,0.50-0.60,14942,0.568933,"[[3158, 4020], [2421, 5343]]",0.439955,0.688176
1,0.60-0.70,19888,0.650141,"[[4087, 4979], [1979, 8843]]",0.450805,0.817132
2,0.70-0.80,14777,0.776206,"[[5993, 1579], [1728, 5477]]",0.791469,0.760167
3,0.80-0.90,20601,0.874521,"[[9609, 1146], [1439, 8407]]",0.893445,0.853849
4,0.90-0.95,15834,0.927056,"[[7522, 491], [664, 7157]]",0.938725,0.915100
5,0.95-0.99,19495,0.959015,"[[8438, 343], [456, 10258]]",0.960938,0.957439
6,0.99-1.00,6410,0.980655,"[[2418, 62], [62, 3868]]",0.975000,0.984224


### E01 결과 해석 — 실행 후 사용자가 작성

- Accuracy 및 baseline delta: Baseline 0.8091061 → E01 0.8091150, 전체 111,947건 중 정답이 90,577 → 90,578로 1건 증가해 사실상 동일한 성능이다.
- speaker=0/1 Recall delta: speaker=0 Recall은 0.764974 → 0.765624로 +0.000650 증가했고, speaker=1 Recall은 0.850005 → 0.849420으로 -0.000585 감소했다. 즉 119대원 Recall의 소폭 개선과 신고자 Recall의 소폭 하락이 서로 상쇄됐다.
- high-confidence error 변화: 2,086 → 2,078, 8건 감소했다. 방향은 긍정적이지만 전체 Validation 규모에 비해 작아 duration의 유의미한 개선 효과라고 판단하기에는 부족하다.
- duration/overlap/confidence slice 관찰: Duration slice에서는 very_short와 short Accuracy가 각각 약 -0.225%p, -0.200%p 하락했고, medium·long은 거의 변하지 않았다. very_long은 약 +0.413%p 개선됐다. 따라서 긴 발화에서 일부 개선 신호는 있으나 전체 성능 향상으로 이어지지는 않았다.
Overlap은 no_overlap 약 +0.032%p, overlap 약 -0.039%p로 사실상 동일했다. Overlap은 평가용 metadata이며 모델 feature에는 포함되지 않았다.
Confidence slice도 구간별 상승·하락이 혼재하며 일관된 개선 패턴은 확인되지 않았다.
- leakage/evaluation mismatch 재점검: Baseline exact reproduction을 먼저 통과했고 동일한 Training/Validation 행, MFCC40, label, scaler/LR 조건에서 duration_ms 한 개만 추가했다. startAt_ms, endAt_ms 절대값은 feature에 포함되지 않았고 overlap도 evaluation-only로 유지됐다. 현재 확인 범위에서 leakage 또는 비교조건 불일치 증거는 없다.
- 다음 결정: EDA에서는 speaker별 duration 차이가 관찰됐지만, MFCC40에 추가했을 때 전체 Accuracy 개선은 사실상 0이었고 두 speaker Recall 사이의 trade-off만 발생했다. 따라서 duration을 후속 feature 조합에 자동 포함하지 않고, 다음 Training-only acoustic candidate screening으로 진행한다.

> 실행 전에는 개선 여부를 작성하지 않는다.

# Checkpoint D — Candidate Feature Screening

1. **무엇:** E02 이후 후보를 미리 고정하지 않고 두 단계로 검토한다.
2. **입력:** Stage 1은 Raw EDA Training 600행, Stage 2는 full Training candidate cache와 `call_id` GroupSplit만 사용한다.
3. **출력 의미:** Stage 1은 계산 가능성/분포 진단일 뿐 순위가 아니며, 후보 우선순위 근거는 Stage 2다.
4. **믿으면 안 되는 경우:** Stage 1 표본을 전체 Training으로 일반화하거나 official Validation을 screening에 사용하면 안 된다.
5. **다음 판단:** Stage 2 후에도 자동 winner는 없으며 사용자가 feature/group을 명시적으로 lock한다.

후보는 모두 waveform에서 직접 계산된다. duration 외 annotation, 순서, absolute timestamp, overlap은 후보 registry에 없다. F0는 현재 Raw EDA cache에서 비활성화되어 있으므로 registry에 포함하지 않았다.

In [7]:
candidate_table = candidate_registry_table()
assert candidate_table['waveform_only'].all()
assert not set(['overlap_status', 'startAt_ms', 'endAt_ms', 'utt_idx']) & set(candidate_table['name'])
display(candidate_table)

,name,family,waveform_only,rule_compliance,information_difference_from_mfcc40,computation_cost
0,silence_ratio,energy_occupancy,True,waveform-derived audio only; no text/order/abs...,summarizes low-energy frame occupancy rather t...,low
1,rms,energy,True,waveform-derived audio only; no text/order/abs...,summarizes utterance-level signal energy,low
2,rms_frame_std,energy,True,waveform-derived audio only; no text/order/abs...,summarizes frame-to-frame energy variability,low
3,peak_amplitude,amplitude,True,waveform-derived audio only; no text/order/abs...,captures peak level not represented explicitly...,low
4,amplitude_dynamic_range,amplitude,True,waveform-derived audio only; no text/order/abs...,captures robust amplitude spread,low
5,clipping_ratio,amplitude_quality,True,waveform-derived audio only; no text/order/abs...,tracks waveform saturation as an audio-quality...,low
6,zcr,zero_crossing,True,waveform-derived audio only; no text/order/abs...,captures sign-change rate related to noisiness...,low
7,spectral_centroid,spectral,True,waveform-derived audio only; no text/order/abs...,explicitly summarizes spectral center of mass,medium
8,spectral_bandwidth,spectral,True,waveform-derived audio only; no text/order/abs...,explicitly summarizes spectral spread,medium
9,spectral_rolloff,spectral,True,waveform-derived audio only; no text/order/abs...,summarizes the upper spectral energy boundary,medium


In [9]:
if not RUN_STAGE1_FEASIBILITY:
    raise RuntimeError('STOP: Stage 1은 필요할 때만 RUN_STAGE1_FEASIBILITY=True로 바꾸세요.')
assert EDA_ACOUSTIC_PATH.is_file() and EDA_ACOUSTIC_META_PATH.is_file()
assert EDA_MANIFEST_META_PATH.is_file()
eda_acoustic_meta = json.loads(EDA_ACOUSTIC_META_PATH.read_text(encoding='utf-8'))
validate_raw_eda_acoustic_metadata(eda_acoustic_meta, eda_manifest_meta)
# Parquet predicate로 Validation 600행은 읽지 않고 Training partition만 연다.
stage1_training_sample = pd.read_parquet(
    EDA_ACOUSTIC_PATH, filters=[('split', '==', 'Training')]
)
assert len(stage1_training_sample) == 600
stage1_diagnostics = stage1_feasibility_diagnostics(
    stage1_training_sample, STAGE1_CANDIDATE_NAMES, expected_training_rows=600
)
assert stage1_diagnostics['scope']['may_rank_or_select_candidates'] is False
display(stage1_diagnostics['feature_summary'])
display(stage1_diagnostics['class_distribution'])
display(stage1_diagnostics['candidate_mfcc_correlations'])
print('STAGE 1 COMPLETE: feasibility diagnostic only; no candidate ranking/selection was produced.')

,feature,dtype,rows,finite_count,missing_count,inf_count,p01,p05,q1,median,q3,p95,p99,class_1_minus_0_smd,max_abs_pearson_with_mfcc40,max_abs_spearman_with_mfcc40,outlier_fraction_beyond_3iqr,computation_cost,rule_compliance,mfcc_information_difference
0,silence_ratio,float64,600,600,0,0,0.000000,0.000000,0.068028,0.150758,0.267857,0.509677,0.755569,-0.085760,0.373530,0.452937,0.005000,low,waveform-derived audio only; no text/order/abs...,summarizes low-energy frame occupancy rather t...
1,rms,float64,600,600,0,0,0.007859,0.017464,0.048912,0.076566,0.121167,0.194031,0.240770,0.234718,0.551247,0.748542,0.001667,low,waveform-derived audio only; no text/order/abs...,summarizes utterance-level signal energy
2,rms_frame_std,float64,600,600,0,0,0.001505,0.002567,0.020792,0.044804,0.069015,0.107714,0.145393,0.215799,0.689670,0.781621,0.000000,low,waveform-derived audio only; no text/order/abs...,summarizes frame-to-frame energy variability
3,peak_amplitude,float64,600,600,0,0,0.044980,0.067043,0.240051,0.500854,0.760437,0.999756,1.000000,-0.054123,0.773830,0.826601,0.000000,low,waveform-derived audio only; no text/order/abs...,captures peak level not represented explicitly...
4,amplitude_dynamic_range,float64,600,600,0,0,0.017080,0.037734,0.089561,0.163330,0.276370,0.436279,0.551338,0.191158,0.639112,0.810523,0.000000,low,waveform-derived audio only; no text/order/abs...,captures robust amplitude spread
5,clipping_ratio,float64,600,600,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000047,0.001089,0.153389,0.112665,0.259426,0.000000,low,waveform-derived audio only; no text/order/abs...,tracks waveform saturation as an audio-quality...
6,zcr,float64,600,600,0,0,0.101493,0.119476,0.152554,0.168471,0.202060,0.258948,0.321726,-0.990410,0.672967,0.614553,0.003333,low,waveform-derived audio only; no text/order/abs...,captures sign-change rate related to noisiness...
7,spectral_centroid,float64,600,600,0,0,623.569133,665.091779,829.795004,980.793963,1109.430175,1358.403281,1539.233602,-1.167899,0.564338,0.497038,0.000000,medium,waveform-derived audio only; no text/order/abs...,explicitly summarizes spectral center of mass
8,spectral_bandwidth,float64,600,600,0,0,70.599847,87.991734,689.867324,768.016690,833.780176,949.871467,1027.718087,-0.534739,0.905377,0.528643,0.115000,medium,waveform-derived audio only; no text/order/abs...,explicitly summarizes spectral spread
9,spectral_rolloff,float64,600,600,0,0,675.210562,680.900476,1425.300537,1720.621056,1961.126710,2412.207031,2691.016996,-0.910423,0.726152,0.499193,0.000000,medium,waveform-derived audio only; no text/order/abs...,summarizes the upper spectral energy boundary


,feature,speaker,finite_count,mean,std,median
0,silence_ratio,0,300,0.194658,0.182253,0.148792
1,silence_ratio,1,300,0.179950,0.159404,0.153394
2,rms,0,300,0.083512,0.052351,0.073623
3,rms,1,300,0.096776,0.060207,0.083552
4,rms_frame_std,0,300,0.044413,0.029016,0.040817
5,rms_frame_std,1,300,0.051864,0.039172,0.048723
6,peak_amplitude,0,300,0.512141,0.284561,0.508301
7,peak_amplitude,1,300,0.495970,0.311407,0.481567
8,amplitude_dynamic_range,0,300,0.183607,0.122276,0.156305
9,amplitude_dynamic_range,1,300,0.209078,0.142966,0.167218


,candidate_feature,mfcc_feature,pearson,spearman
0,silence_ratio,mfcc_01_mean,-0.098834,-0.289210
1,silence_ratio,mfcc_02_mean,-0.034634,0.082515
2,silence_ratio,mfcc_03_mean,0.179081,0.165242
3,silence_ratio,mfcc_04_mean,0.332991,0.336242
4,silence_ratio,mfcc_05_mean,0.373530,0.415104
...,...,...,...,...
435,spectral_flatness,mfcc_16_std,0.049700,0.131919
436,spectral_flatness,mfcc_17_std,0.155317,0.153342
437,spectral_flatness,mfcc_18_std,0.090672,0.159605
438,spectral_flatness,mfcc_19_std,0.046632,0.130334


STAGE 1 COMPLETE: feasibility diagnostic only; no candidate ranking/selection was produced.


### Stage 1 검토 기록 — 실행 후 작성

- 계산 불가 또는 missing/inf 후보: 없음. 11개 후보 모두 Training 600행에서 finite_count=600, missing=0, inf=0이라 계산 안정성 문제는 확인되지 않았다.
- outlier 민감도가 큰 후보: spectral_bandwidth가 outlier_fraction_beyond_3iqr = 0.115로 가장 크다. 나머지는 대부분 0~0.5% 수준이라 상대적으로 안정적이다. 따라서 bandwidth는 full Training에서 분포를 다시 확인할 필요가 있다.
- MFCC40과 redundancy가 높아 추가 검토가 필요한 후보: spectral_bandwidth: max Pearson 0.905, peak_amplitude: max Spearman 0.827, amplitude_dynamic_range: max Spearman 0.811, rms_frame_std: max Spearman 0.782, rms: max Spearman 0.749
는 기존 MFCC40과 정보 중복 가능성이 상대적으로 높다.
이건 “쓸모없다”는 뜻이 아니라 incremental signal이 작을 가능성이 있어 우선순위를 낮출 근거다.
- full Training cache에서 다시 검증할 후보와 계산 비용:

| 후보                  | Stage 1 관찰                          | 판단                                                 |
| ------------------- | ----------------------------------- | -------------------------------------------------- |
| `spectral_centroid` | SMD `-1.168`, MFCC corr 중간 수준       | **가장 검증 가치 큼**                                     |
| `zcr`               | SMD `-0.990`, MFCC corr 약 0.61~0.67 | **강한 후보**                                          |
| `spectral_rolloff`  | SMD `-0.910`                        | 강한 차이, 다만 MFCC 중복은 centroid보다 큼                    |
| `spectral_flatness` | SMD `-0.764`, corr 약 0.47~0.53      | **상대적으로 독립적인 신호 가능성**                              |
| `silence_ratio`     | SMD `-0.086`로 약함                    | class 차이는 약하지만 MFCC와 중복이 비교적 작고 계산비용 낮아서 대조 후보로 유지 |



> 이 기록은 후보의 최종 순위 또는 성능 개선 증거가 아니다.

In [ ]:
assert globals().get('BASELINE_GATE_PASSED', False)
if not BUILD_FULL_TRAINING_CANDIDATE_CACHE:
    raise RuntimeError('STOP: full Training audio extraction은 기본 비활성화입니다.')
if not FULL_TRAINING_CACHE_FEATURES:
    raise RuntimeError('Stage 1 검토 후 FULL_TRAINING_CACHE_FEATURES를 명시하세요.')
training_candidate_paths = build_candidate_feature_cache(
    full_eda_manifest,
    data_root=DATA_ROOT,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    feature_names=FULL_TRAINING_CACHE_FEATURES,
    source_signature=eda_manifest_meta['source_signature'],
    allow_full_extraction=True,
    on_call_done=lambda done, total: print(f'Training calls: {done:,}/{total:,}') if done % 500 == 0 or done == total else None,
)
print('Saved immutable Training candidate cache:', training_candidate_paths)

In [ ]:
if not RUN_STAGE2_TRAINING_SCREEN:
    raise RuntimeError('STOP: Training cache와 CANDIDATE_GROUPS를 검토한 뒤 switch를 켜세요.')
if not FULL_TRAINING_CACHE_FEATURES or not CANDIDATE_GROUPS:
    raise RuntimeError('FULL_TRAINING_CACHE_FEATURES와 CANDIDATE_GROUPS를 먼저 정의하세요.')
if not set(feature for group in CANDIDATE_GROUPS.values() for feature in group).issubset(FULL_TRAINING_CACHE_FEATURES):
    raise AssertionError('CANDIDATE_GROUPS contains a feature absent from the Training cache contract')
training_candidate_cache = load_candidate_feature_cache(
    full_eda_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    feature_names=FULL_TRAINING_CACHE_FEATURES,
    source_signature=eda_manifest_meta['source_signature'],
)
stage2_config = {
    'mfcc': MFCC_CONFIG,
    'reference_model': REFERENCE_MODEL_CONFIG,
    'candidate_extraction': CANDIDATE_EXTRACTION_CONFIG,
}
stage2_result = stage2_training_group_screen(
    aligned_training,
    training_candidate_cache,
    CANDIDATE_GROUPS,
    seed=42,
    test_size=0.2,
    source_signature=eda_manifest_meta['source_signature'],
    config=stage2_config,
)
assert stage2_result['scope']['official_validation_used'] is False
assert int(stage2_result['scope']['source_rows']) == 889358
assert int(stage2_result['split_audit'].loc[0, 'call_id_overlap']) == 0
display(stage2_result['split_audit'])
display(pd.DataFrame([stage2_result['baseline_metrics']]))
display(stage2_result['candidate_performance'])
display(stage2_result['distribution_diagnostics'])
display(stage2_result['redundancy_diagnostics'])
print('STAGE 2 COMPLETE: no automatic winner was selected.')

In [ ]:
if not LOCK_STAGE2_SELECTION:
    raise RuntimeError('STOP: E03/E04 precommit과 선택값을 확인한 뒤에만 selection을 lock하세요.')
if not SELECTED_CANDIDATE_NAME or not SELECTED_FEATURE_NAMES or not SELECTED_EXPERIMENT_ID:
    raise RuntimeError('SELECTED_EXPERIMENT_ID/CANDIDATE_NAME/FEATURE_NAMES를 모두 명시하세요.')
expected_feature_names = PRECOMMITTED_EXPERIMENTS.get(SELECTED_EXPERIMENT_ID)
if expected_feature_names is None:
    raise AssertionError('현재 precommit은 E03/E04만 허용합니다. E05 또는 조합을 만들지 마세요.')
if tuple(SELECTED_FEATURE_NAMES) != expected_feature_names:
    raise AssertionError('선택 feature가 E03/E04 Training-only precommit과 다릅니다.')
if SELECTED_CANDIDATE_NAME != expected_feature_names[0] or len(expected_feature_names) != 1:
    raise AssertionError('E03/E04는 한 번에 하나의 사전 확정 candidate만 실행합니다.')
# Historical v1 summary above is context only. A future v2 lock requires the
# actual Stage 2 result from this kernel; constants cannot supply its evidence.
frozen_precommit = None
CANDIDATE_SELECTION_LOCKED = False
if 'stage2_result' not in globals():
    raise RuntimeError('STOP: run Training-only Stage 2 before freezing a v2 precommit.')
stage2_config = {
    'mfcc': MFCC_CONFIG,
    'reference_model': REFERENCE_MODEL_CONFIG,
    'candidate_extraction': CANDIDATE_EXTRACTION_CONFIG,
}
frozen_precommit = freeze_stage2_precommit(
    stage2_result,
    experiment_id=SELECTED_EXPERIMENT_ID,
    candidate_name=SELECTED_CANDIDATE_NAME,
    feature_names=SELECTED_FEATURE_NAMES,
    config=stage2_config,
    source_signature=eda_manifest_meta['source_signature'],
)
locked_candidate = lock_candidate_selection(
    stage2_result,
    selection_name=frozen_precommit.candidate_name,
    feature_names=frozen_precommit.feature_names,
)
CANDIDATE_SELECTION_LOCKED = True
print('Frozen future-v2 Training-only Stage 2 precommit:', frozen_precommit)


# Checkpoint E — Screening Diagnostics

1. **무엇:** 후보의 dtype, missing/inf, class 분포, MFCC redundancy, split stability, outlier, 비용을 함께 본다.
2. **입력:** Stage 1 Training sample 진단과 Stage 2 full Training internal split 진단.
3. **출력 의미:** 단일 Pearson 값으로 제거하지 않고 여러 관점의 불확실성을 기록한다.
4. **믿으면 안 되는 경우:** univariate separation을 원인 또는 official 성능 개선으로 해석하면 안 된다.
5. **다음 판단:** 실제 후보 선택은 Accuracy 중심 Stage 2 결과와 비용·안정성·규정 적합성을 함께 보고 사용자가 한다.

In [ ]:
if 'stage1_diagnostics' in globals():
    display(stage1_diagnostics['feature_summary'])
if 'stage2_result' in globals():
    display(stage2_result['distribution_diagnostics'])
    display(stage2_result['redundancy_diagnostics'])
    display(stage2_result['candidate_performance'])
if 'stage1_diagnostics' not in globals() and 'stage2_result' not in globals():
    raise RuntimeError('Checkpoint D의 Stage 1 또는 Stage 2를 먼저 실행하세요.')

## E02 실행 결과와 E03/E04 사전 확정 기록

> 아래 E02 수치와 비교는 과거 v1 결과다. 현재 v2 코드로 재평가하지 않았다.

E02=`MFCC40 + spectral_rolloff`의 실제 official Validation 결과는 Accuracy `0.8115090176601427`, baseline delta `+0.0024029228116876356`, confusion matrix `[[41459, 12386], [8715, 49387]]`, recall_0 `0.7699693564862105`(delta `+0.004995821339028761`), recall_1 `0.8500051633334481`(delta `0.0`), high-confidence errors `2123`(delta `+37`)이다. Training-only Stage 2와 official Validation에서 같은 방향의 modest improvement가 재현되어 승격 후보로 남기되, high-confidence error 증가는 별도 위험으로 유지한다. 이 기록은 실제 E02 결과의 요약이며, 디스크에 없는 E02 report 또는 Checkpoint G row를 하드코딩해 복구하지 않는다.

고정 `call_id` GroupSplit(`seed=42`, `test_size=0.2`)은 Training 889,358행을 internal train 711,467행/23,360 calls와 internal validation 177,891행/5,840 calls로 나눴고 call overlap은 0, official Validation 사용은 `False`였다. MFCC40 internal baseline은 Accuracy `0.813397`, recall_0 `0.767431`, recall_1 `0.855792`였다. 단일 후보 Accuracy delta 순서는 `spectral_rolloff +0.001979`, `zcr +0.001180`, `spectral_centroid +0.001152`, `spectral_flatness +0.000416`, `silence_ratio -0.000208`였다.

따라서 **E03=`MFCC40 + zcr`와 E04=`MFCC40 + spectral_centroid`는 각 후보의 official Validation 결과를 보기 전에 Training-only Stage 2 Accuracy 순위로 사전 확정했다.** E02 official Validation 결과를 보고 새로 선택한 것이 아니다. E03 결과가 기대보다 낮아도 E04를 취소하거나 다른 후보로 바꾸지 않는다. 예외는 leakage, cache/evaluation mismatch, preprocessing 오류 등 중단 조건이 발견되는 경우뿐이다. `spectral_flatness`와 `silence_ratio`는 현재 official Validation 후속 실험에서 보류하며 영구 기각하지 않는다.

### E03 수동 config 예시

```python
SELECTED_EXPERIMENT_ID = 'E03'
SELECTED_CANDIDATE_NAME = 'zcr'
SELECTED_FEATURE_NAMES = ('zcr',)
LOCK_STAGE2_SELECTION = True
BUILD_VALIDATION_CANDIDATE_CACHE = False  # 정상 cache가 없을 때만 True
RUN_SELECTED_FEATURE_EXPERIMENT = False   # cache 검증 뒤에만 True
```

### E04 수동 config 예시

```python
SELECTED_EXPERIMENT_ID = 'E04'
SELECTED_CANDIDATE_NAME = 'spectral_centroid'
SELECTED_FEATURE_NAMES = ('spectral_centroid',)
LOCK_STAGE2_SELECTION = True
BUILD_VALIDATION_CANDIDATE_CACHE = False  # 정상 cache가 없을 때만 True
RUN_SELECTED_FEATURE_EXPERIMENT = False   # cache 검증 뒤에만 True
```

두 설정을 동시에 실행하지 않는다. E05, 조합 feature, ensemble, winner는 E03/E04 실제 결과를 모두 검토하기 전에는 생성하거나 선택하지 않는다.

# Checkpoint F — Generic Single-Feature Experiment Runner

1. **무엇:** Training-only Stage 2에서 사전 확정한 E03 또는 E04 단일 feature 하나를 동일 LR로 공식 평가한다.
2. **입력:** locked Training candidate cache와 그 이후 별도로 만든 Validation candidate cache.
3. **출력 의미:** baseline과 같은 report를 반환하지만 experiment success는 Accuracy가 primary다.
4. **믿으면 안 되는 경우:** selection lock 전에 Validation을 만들거나, cache 계약/key/fair-comparison guard가 실패하면 중단한다.
5. **다음 판단:** E03/E04를 모두 실행·검토한 뒤에만 조합 여부를 별도 결정한다. 현재는 조합을 선택하거나 실행하지 않는다.

Validation cache는 screening/랭킹에 사용하지 않는다. overlap은 이 단계에서도 평가용 slice일 뿐 model input이 아니다.

In [ ]:
assert globals().get('CANDIDATE_SELECTION_LOCKED', False), 'Training-only Stage 2 selection is not locked'
active_precommit = require_stage2_precommit(
    globals().get('frozen_precommit'),
    experiment_id=SELECTED_EXPERIMENT_ID,
    candidate_name=SELECTED_CANDIDATE_NAME,
    feature_names=SELECTED_FEATURE_NAMES,
    config=stage2_config,
    source_signature=eda_manifest_meta['source_signature'],
)
# Full Training 5-feature cache는 strict metadata/source/key/finite 검증 후 그대로 재사용한다.
training_candidate_cache = load_candidate_feature_cache(
    full_eda_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Training',
    feature_names=FULL_TRAINING_CACHE_FEATURES, source_signature=eda_manifest_meta['source_signature']
)
print('Validated/reused immutable Training candidate cache:', tuple(training_candidate_cache.columns))

validation_candidate_paths = candidate_cache_paths(
    CANDIDATE_CACHE_ROOT, 'Validation', active_precommit.feature_names
)
validation_cache_exists = tuple(path.is_file() for path in validation_candidate_paths)
if all(validation_cache_exists):
    validation_candidate_cache = load_candidate_feature_cache(
        full_eda_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Validation',
        feature_names=active_precommit.feature_names, source_signature=eda_manifest_meta['source_signature']
    )
    print('Validated/reused immutable Validation candidate cache:', validation_candidate_paths)
elif any(validation_cache_exists):
    raise AssertionError(f'partial Validation candidate cache; do not overwrite: {validation_candidate_paths}')
else:
    if not BUILD_VALIDATION_CANDIDATE_CACHE:
        raise RuntimeError('STOP: locked candidate cache가 없습니다. 검토 후 BUILD_VALIDATION_CANDIDATE_CACHE=True로 바꾸세요.')
    validation_candidate_paths = build_candidate_feature_cache(
        full_eda_manifest,
        data_root=DATA_ROOT,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Validation',
        feature_names=active_precommit.feature_names,
        source_signature=eda_manifest_meta['source_signature'],
        allow_full_extraction=True,
        on_call_done=lambda done, total: print(f'Validation calls: {done:,}/{total:,}') if done % 250 == 0 or done == total else None,
    )
    validation_candidate_cache = load_candidate_feature_cache(
        full_eda_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Validation',
        feature_names=active_precommit.feature_names, source_signature=eda_manifest_meta['source_signature']
    )
    print('Built and validated immutable Validation candidate cache:', validation_candidate_paths)

In [11]:
assert globals().get('BASELINE_GATE_PASSED', False)
assert globals().get('CANDIDATE_SELECTION_LOCKED', False)
active_precommit = require_stage2_precommit(
    globals().get('frozen_precommit'),
    experiment_id=SELECTED_EXPERIMENT_ID,
    candidate_name=SELECTED_CANDIDATE_NAME,
    feature_names=SELECTED_FEATURE_NAMES,
    config=stage2_config,
    source_signature=eda_manifest_meta['source_signature'],
)
assert 'training_candidate_cache' in globals() and 'validation_candidate_cache' in globals(), 'Checkpoint F cache validation을 먼저 실행하세요.'
if not RUN_SELECTED_FEATURE_EXPERIMENT:
    raise RuntimeError('STOP: cache/selection을 검토한 뒤 RUN_SELECTED_FEATURE_EXPERIMENT=True로 바꾸세요.')
if SELECTED_EXPERIMENT_ID not in PRECOMMITTED_EXPERIMENTS:
    raise AssertionError('현재 실행 가능한 precommit은 E03/E04뿐입니다.')
if active_precommit.feature_names != PRECOMMITTED_EXPERIMENTS[SELECTED_EXPERIMENT_ID]:
    raise AssertionError('locked candidate와 selected experiment precommit이 다릅니다.')
if 'executed_feature_reports' not in globals():
    executed_feature_reports = {}
if 'selected_report' in globals():
    executed_feature_reports.setdefault(selected_report['experiment_id'], selected_report)
existing_experiment_ids = set(executed_feature_reports)
if 'experiment_table' in globals():
    existing_experiment_ids.update(experiment_table['experiment_id'].astype(str))
if active_precommit.experiment_id in existing_experiment_ids:
    raise AssertionError(f'experiment ID already executed/recorded: {SELECTED_EXPERIMENT_ID}')
selected_result = run_feature_experiment(
    experiment_id=active_precommit.experiment_id,
    training=aligned_training,
    validation=aligned_validation,
    extra_training=training_candidate_cache,
    extra_validation=validation_candidate_cache,
    extra_feature_names=active_precommit.feature_names,
    baseline_result=baseline_result,
)
selected_report = build_experiment_report(
    selected_result,
    training_tracking=aligned_training.tracking,
    evaluation_tracking=aligned_validation.tracking,
    baseline_report=baseline_report,
)
executed_feature_reports[active_precommit.experiment_id] = selected_report
print_experiment_report(selected_report)
display(selected_report['duration_summary'])
display(selected_report['overlap_summary'])
display(selected_report['confidence_summary'])

Experiment ID: E05
Feature names: ['mfcc_01_mean', 'mfcc_02_mean', 'mfcc_03_mean', 'mfcc_04_mean', 'mfcc_05_mean', 'mfcc_06_mean', 'mfcc_07_mean', 'mfcc_08_mean', 'mfcc_09_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'mfcc_14_mean', 'mfcc_15_mean', 'mfcc_16_mean', 'mfcc_17_mean', 'mfcc_18_mean', 'mfcc_19_mean', 'mfcc_20_mean', 'mfcc_01_std', 'mfcc_02_std', 'mfcc_03_std', 'mfcc_04_std', 'mfcc_05_std', 'mfcc_06_std', 'mfcc_07_std', 'mfcc_08_std', 'mfcc_09_std', 'mfcc_10_std', 'mfcc_11_std', 'mfcc_12_std', 'mfcc_13_std', 'mfcc_14_std', 'mfcc_15_std', 'mfcc_16_std', 'mfcc_17_std', 'mfcc_18_std', 'mfcc_19_std', 'mfcc_20_std', 'spectral_rolloff', 'zcr']
Feature dimension: 42
Training shape: (889358, 42)
Validation shape: (111947, 42)
Accuracy: 0.8131169214003054
Confusion matrix: [[41697, 12148], [8773, 49329]]
speaker=0 Recall: 0.7743894512025258
speaker=1 Recall: 0.8490069188668204
Accuracy delta vs baseline: 0.004010826551850366
Recall 0 delta vs baseline: 0.0094

,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,very_short,5786,0.787245,"[[2347, 531], [700, 2208]]",0.815497,0.759285
1,short,22545,0.757818,"[[7051, 2673], [2787, 10034]]",0.725113,0.782622
2,medium,55659,0.802314,"[[22661, 7179], [3824, 21995]]",0.759417,0.851892
3,long,22384,0.876921,"[[8170, 1550], [1205, 11459]]",0.840535,0.904848
4,very_long,5573,0.915306,"[[1468, 215], [257, 3633]]",0.872252,0.933933


,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,no_overlap,63084,0.847442,"[[24308, 5578], [4046, 29152]]",0.813357,0.878125
1,overlap,48863,0.768803,"[[17389, 6570], [4727, 20177]]",0.725782,0.810191


,slice,sample_count,accuracy,confusion_matrix,recall_0,recall_1
0,0.50-0.60,14208,0.567497,"[[2884, 3704], [2441, 5179]]",0.437766,0.679659
1,0.60-0.70,20253,0.650867,"[[4047, 5083], [1988, 9135]]",0.443264,0.821271
2,0.70-0.80,14238,0.788383,"[[6133, 1312], [1701, 5092]]",0.823774,0.749595
3,0.80-0.90,20762,0.877950,"[[10450, 1065], [1469, 7778]]",0.907512,0.841138
4,0.90-0.95,16094,0.926805,"[[8249, 501], [677, 6667]]",0.942743,0.907816
5,0.95-0.99,19114,0.956315,"[[8278, 390], [445, 10001]]",0.955007,0.957400
6,0.99-1.00,7278,0.980077,"[[1656, 93], [52, 5477]]",0.946827,0.990595


### E03 결과 해석 — 실행 후 사용자가 작성

- Accuracy 및 baseline delta:  
  Official Validation Accuracy는 `0.8101155011`로, E00 baseline `0.8091060948` 대비 `+0.0010094062` (`+0.101%p`) 개선되었다. 정답 수 기준 net `+113건`이다. 작은 폭이지만 Training-only Stage 2에서 관찰한 개선 방향과 일치했다.

- speaker=0/1 Recall delta:  
  speaker=0 Recall은 `0.7691521961`로 baseline 대비 `+0.0041786610` (`+0.418%p`) 개선되었다.  
  speaker=1 Recall은 `0.8480775188`로 baseline 대비 `-0.0019276445` (`-0.193%p`) 하락했다.  
  따라서 zcr 추가 효과는 class별로 균형적이지 않고, speaker=0 개선과 speaker=1 하락의 trade-off가 관찰되었다.

- high-confidence error 변화:  
  High-confidence error는 baseline `2,086건`에서 `2,102건`으로 `+16건` 증가했다. Accuracy는 개선됐지만 고확신 오답까지 함께 개선된 것은 아니다.

- duration/overlap/confidence slice 관찰:  
  Duration Accuracy는 very_short `0.787764`, short `0.754802`, medium `0.799386`, long `0.872588`, very_long `0.913332`였다.  
  no_overlap Accuracy는 `0.843954`, overlap Accuracy는 `0.766429`였다.  
  전체적으로 baseline 대비 일부 변화는 있으나 특정 slice에서 일관된 대폭 개선이 확인된 것은 아니다.

- leakage/evaluation mismatch 재점검:  
  E00과 동일한 MFCC40, StandardScaler Training-only fit, LogisticRegression 설정, float32 numerical path, official Validation 평가 계약을 사용했다. 추가 입력은 허용된 waveform-derived `zcr` 하나이며 text, utterance order, absolute timestamp, overlap 등 금지 정보는 model feature에 포함되지 않았다. 비정상적인 성능 상승도 관찰되지 않았다.

- 다음 결정:  
  zcr는 baseline 대비 작은 추가 signal을 보였으므로 보조 candidate로 유지한다. 다만 speaker=1 Recall 하락과 high-confidence error 증가가 있어 단일 feature 최우선 후보로 보기는 어렵다. 사전 확정된 E04 `spectral_centroid` 실험은 E03 결과와 관계없이 계획대로 진행한다.

> 실행 전에는 개선 여부를 작성하지 않는다. E03 결과가 기대보다 낮아도 precommitted E04 후보를 바꾸지 않는다.

### E04 결과 해석 — 실행 후 사용자가 작성

- Accuracy 및 baseline delta:  
  Official Validation Accuracy는 `0.8104996114`로, E00 baseline `0.8091060948` 대비 `+0.0013935166` (`+0.139%p`) 개선되었다. 정답 수 기준 net `+156건`이다. 작은 폭이지만 baseline보다 일관된 개선이 확인됐다.

- speaker=0/1 Recall delta:  
  speaker=0 Recall은 `0.7658649828`로 baseline 대비 `+0.0008914477` (`+0.089%p`) 개선되었다.  
  speaker=1 Recall은 `0.8518639634`로 baseline 대비 `+0.0018588000` (`+0.186%p`) 개선되었다.  
  E03와 달리 두 class Recall이 모두 상승해 상대적으로 더 균형적인 변화가 관찰됐다.

- high-confidence error 변화:  
  High-confidence error는 baseline `2,086건`에서 `2,102건`으로 `+16건` 증가했다. Accuracy와 양쪽 Recall은 개선됐지만 고확신 오답까지 함께 줄어든 것은 아니다.

- duration/overlap/confidence slice 관찰:  
  Duration Accuracy는 very_short `0.790702`, short `0.756576`, medium `0.798397`, long `0.873570`, very_long `0.916741`이었다.  
  no_overlap Accuracy는 `0.844905`, overlap Accuracy는 `0.766081`이었다.  
  일부 duration slice에서 baseline보다 개선됐지만 모든 slice에서 일관된 큰 개선이 나타난 것은 아니다. Confidence 구간에서도 전체적인 구조 변화보다는 작은 성능 변화로 해석하는 것이 적절하다.

- leakage/evaluation mismatch 재점검:  
  E00과 동일한 MFCC40, StandardScaler Training-only fit, LogisticRegression 설정, float32 numerical path, official Validation 평가 계약을 유지했다. 추가 입력은 허용된 waveform-derived `spectral_centroid` 하나이며 text, utterance order, absolute timestamp, overlap 등 금지 정보는 model feature에 포함되지 않았다. 비정상적으로 큰 성능 상승도 관찰되지 않았다.

- 다음 결정:  
  `spectral_centroid`는 baseline 대비 작은 추가 signal을 보였고 speaker=0/1 Recall이 모두 개선되어 E03보다 class balance 측면에서는 더 안정적이었다. 다만 단일-feature Accuracy는 E02 `spectral_rolloff`보다 낮고 high-confidence error도 감소하지 않았다. 따라서 단일-feature 최우선 후보는 E02를 유지하고, E04는 조합 후보 판단에 사용할 유효 candidate로 유지한다.

> 실행 전에는 개선 여부를 작성하지 않는다. E04 결과까지 확인한 뒤에만 단일-feature 후보를 비교하고, 조합 여부는 별도 Training-only 진단 후 결정한다.

# Checkpoint G — Experiment Decision Table

1. **무엇:** 실제로 실행된 baseline/candidate report만 동일 schema에 누적한다.
2. **입력:** 현재 kernel에 존재하는 실제 report 객체.
3. **출력 의미:** 각 행은 해당 run prediction에서 계산된 값이며 빈 상태에서는 metric 행이 없다.
4. **믿으면 안 되는 경우:** 수동으로 가짜 수치를 넣거나 다른 evaluation set 결과를 같은 표에 섞으면 안 된다.
5. **다음 판단:** Accuracy 우선으로 비교하고 slice는 오류 변화 설명에만 사용한다.

In [12]:
if 'experiment_table' not in globals():
    experiment_table = empty_experiment_table()
reports_to_append = []
if 'baseline_report' in globals():
    reports_to_append.append(baseline_report)
if 'e01_report' in globals():
    reports_to_append.append(e01_report)
if 'executed_feature_reports' in globals():
    reports_to_append.extend(executed_feature_reports.values())
elif 'selected_report' in globals():
    reports_to_append.append(selected_report)
for observed_report in reports_to_append:
    observed_id = str(observed_report['experiment_id'])
    if observed_id not in set(experiment_table['experiment_id'].astype(str)):
        experiment_table = append_comparison_row(experiment_table, observed_report)
display(experiment_table)
if SAVE_DECISION_TABLE:
    DECISION_TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    experiment_table.to_csv(DECISION_TABLE_PATH, index=False, encoding='utf-8-sig')
    print('Saved:', DECISION_TABLE_PATH)
else:
    print('SAVE_DECISION_TABLE=False: table remains in memory only.')

,experiment_id,baseline_or_candidate,added_feature,train_rows,validation_rows,feature_dim,accuracy,accuracy_delta,recall_0,recall_1,recall_0_delta,recall_1_delta,high_conf_error_count,overlap_accuracy,short_accuracy,rule_compliance,notes
0,M2-REFERENCE-MFCC40-LR,baseline,,889358,111947,40,0.809106,0.0,0.764974,0.850005,0.0,0.0,2086,0.765426,0.755068,validated_by_runtime_guards,
1,E05,candidate,"spectral_rolloff, zcr",889358,111947,42,0.813117,0.004011,0.774389,0.849007,0.009416,-0.000998,2158,0.768803,0.757818,validated_by_runtime_guards,


SAVE_DECISION_TABLE=False: table remains in memory only.


## E03/E04 실행 및 이후 결정 원칙

- E03=`zcr`, E04=`spectral_centroid`는 official Validation을 보기 전에 Training-only Stage 2 Accuracy 순위로 사전 확정했다.
- E03 결과가 좋거나 나쁘다는 이유로 E04를 취소하거나 다른 후보로 교체하지 않는다.
- E03과 E04는 한 번에 하나씩 수동으로 selection/lock/cache validation/run한다.
- 각 primary comparison은 항상 frozen E00 baseline 대비 delta이며, E02와의 비교는 E03/E04 모두 완료된 뒤 secondary comparison으로만 수행한다.
- `spectral_flatness`와 `silence_ratio`는 현재 official Validation 후속 실험에서 보류하며 영구 기각하지 않는다.
- E03/E04 모두 실제 실행되기 전에는 E05, 조합 feature, ensemble, winner를 생성하거나 선택하지 않는다.
- Accuracy가 개선되지 않으면 slice 개선만으로 최종 개선이라 부르지 않는다.
- 큰 상승이 나오면 성능 해석 전에 leakage, preprocessing/evaluation mismatch, target proxy, feature alignment를 먼저 점검한다.

## 즉시 중단하고 가져올 조건

- baseline Notebook code hash 또는 cache JSON 계약 불일치
- Training/Validation row 수, 40차원 MFCC, label 0/1, call 수 불일치
- record_key 중복, cache↔manifest join 누락, start/end/duration/label 불일치
- exact confusion matrix `[[41190, 12655], [8715, 49387]]` 불일치
- E01이 41차원이 아니거나 앞 40개 MFCC 값이 달라짐
- E03/E04가 각각 41차원이 아니거나 앞 40개 MFCC가 bitwise 동일하지 않음
- E03/E04 model matrix 또는 scaled matrix의 float32 numerical path 불일치
- scaler/model fit scope가 Training이 아님
- Stage 2 call_id overlap이 0이 아님
- candidate cache metadata/source/key signature 불일치 또는 missing/inf
- feature name에 absolute timestamp, order/id/path, overlap/quality metadata가 등장
- selected experiment/candidate/feature tuple이 E03/E04 precommit과 다르거나 experiment ID가 이미 존재함
- 예상보다 큰 성능 상승 또는 baseline과 다른 preprocessing/model 설정 발견

위 조건에서는 cache를 덮어쓰거나 switch를 우회하지 않는다. 큰 상승은 개선으로 해석하기 전에 leakage, preprocessing mismatch, evaluation mismatch, target proxy, feature alignment를 먼저 확인한다.

# Checkpoint H — Training-only 조합 전 진단

목표는 조합 모델의 성능 평가가 아니라, 기존 단일-feature 후보 사이의 **redundancy(중복성)**와 **prediction complementarity(예측 보완성)**를 동일한 Training-only Stage 2 fixed `call_id` split에서 진단하는 것이다.

- 대상: `spectral_rolloff` = Spectral Rolloff(스펙트럼 롤오프 주파수), `zcr` = Zero Crossing Rate(영교차율), `spectral_centroid` = Spectral Centroid(스펙트럼 중심)
- 입력: Training 889,358행의 frozen MFCC40, immutable 5-feature Training candidate cache, 이미 생성된 Stage 2 train/internal-validation indices
- 제외: `duration_ms`, `spectral_flatness`, `silence_ratio`, official Validation feature/prediction/error/metric/cache
- fit 허용 범위: MFCC40 baseline 1개와 MFCC40+단일 candidate 3개만 Training internal train에서 fit
- 금지: 2-feature 조합 fit, E05 생성, 자동 winner/추천 조합 출력

E00~E04 official Validation 결과는 기록용 context일 뿐 이 진단의 입력이 아니다. 높은 correlation은 redundancy 가능성을 나타내는 진단 근거이지 자동 제거 규칙이 아니다. correlation 하나만으로 조합을 결정하지 않는다.


In [60]:
from itertools import combinations

import numpy as np
import pandas as pd

from dcc_data.mission2_experiments import (
    BASELINE_FEATURE_NAMES,
    align_extra_feature_matrix,
    assemble_experiment_matrix,
    assert_single_factor_comparison,
    fit_reference_logistic_regression,
)


RUN_TRAINING_ONLY_COMBINATION_DIAGNOSTICS = False

COMBINATION_DIAGNOSTIC_FEATURES = (
    'spectral_rolloff',
    'zcr',
    'spectral_centroid',
)
COMBINATION_DIAGNOSTIC_PAIRS = tuple(combinations(COMBINATION_DIAGNOSTIC_FEATURES, 2))
FIXED_STAGE2_SPLIT_CONTRACT = {
    'seed': 42,
    'test_size': 0.2,
    'train_rows': 711_467,
    'internal_validation_rows': 177_891,
    'train_calls': 23_360,
    'internal_validation_calls': 5_840,
    'call_id_overlap': 0,
}

CANDIDATE_PAIR_REDUNDANCY_COLUMNS = (
    'feature_a',
    'feature_b',
    'internal_train_rows',
    'internal_validation_rows',
    'train_pearson',
    'internal_val_pearson',
    'train_spearman',
    'internal_val_spearman',
    'pearson_same_direction',
    'spearman_same_direction',
    'pearson_abs_difference',
    'spearman_abs_difference',
)
CANDIDATE_PAIR_COMPLEMENTARITY_COLUMNS = (
    'feature_a',
    'feature_b',
    'internal_validation_rows',
    'both_correct',
    'both_wrong',
    'a_correct_b_wrong',
    'a_wrong_b_correct',
    'prediction_disagreement_count',
    'prediction_disagreement_rate',
    'error_intersection_count',
    'error_union_count',
    'error_jaccard',
    'speaker_0_a_correct_b_wrong',
    'speaker_0_a_wrong_b_correct',
    'speaker_1_a_correct_b_wrong',
    'speaker_1_a_wrong_b_correct',
)
BASELINE_ERROR_CHANGE_COLUMNS = (
    'candidate',
    'internal_validation_rows',
    'baseline_wrong_candidate_correct',
    'baseline_correct_candidate_wrong',
    'net_correct_gain',
)


def _binary_vector(values, *, name, expected_rows=None):
    vector = np.asarray(values, dtype=np.int64)
    if vector.ndim != 1:
        raise AssertionError(f'{name} must be one-dimensional')
    if expected_rows is not None and len(vector) != expected_rows:
        raise AssertionError(f'{name} row mismatch: {len(vector)} vs {expected_rows}')
    if not set(np.unique(vector)).issubset({0, 1}):
        raise AssertionError(f'{name} must contain only 0/1')
    return vector


def _validated_partition_indices(indices, *, name, row_count):
    positions = np.asarray(indices, dtype=np.int64)
    if positions.ndim != 1 or len(positions) == 0:
        raise AssertionError(f'{name} must be a non-empty one-dimensional index')
    if len(np.unique(positions)) != len(positions):
        raise AssertionError(f'{name} contains duplicate positions')
    if positions.min() < 0 or positions.max() >= row_count:
        raise AssertionError(f'{name} contains an out-of-range position')
    return positions


def validate_training_only_combination_diagnostic_scope(
    *,
    source_split,
    source_rows,
    official_validation_used,
    split_audit,
    feature_names,
    feature_pairs,
):
    if str(source_split) != 'Training' or official_validation_used is not False:
        raise AssertionError('Training-only diagnostic cannot use official Validation')
    if int(source_rows) != 889_358:
        raise AssertionError(f'Training source row mismatch: {source_rows}')
    actual_audit = {
        'seed': int(split_audit['seed']),
        'test_size': float(split_audit['test_size']),
        'train_rows': int(split_audit['train_rows']),
        'internal_validation_rows': int(split_audit['internal_validation_rows']),
        'train_calls': int(split_audit['train_calls']),
        'internal_validation_calls': int(split_audit['internal_validation_calls']),
        'call_id_overlap': int(split_audit['call_id_overlap']),
    }
    if actual_audit != FIXED_STAGE2_SPLIT_CONTRACT:
        raise AssertionError(
            f'fixed Stage 2 split contract mismatch: {actual_audit!r}'
        )
    if tuple(feature_names) != COMBINATION_DIAGNOSTIC_FEATURES:
        raise AssertionError('diagnostic feature set must remain the three predeclared candidates')
    normalized_pairs = tuple(tuple(pair) for pair in feature_pairs)
    if normalized_pairs != COMBINATION_DIAGNOSTIC_PAIRS:
        raise AssertionError('diagnostic pair set must remain the three predeclared pairs')


def _same_correlation_direction(left, right):
    return bool(
        np.isfinite(left)
        and np.isfinite(right)
        and np.sign(left) == np.sign(right)
    )


def _absolute_difference(left, right):
    return float(abs(left - right)) if np.isfinite(left) and np.isfinite(right) else np.nan


def build_candidate_pair_redundancy_table(
    candidate_values,
    train_indices,
    internal_validation_indices,
):
    frame = pd.DataFrame(candidate_values).reset_index(drop=True)
    missing = sorted(set(COMBINATION_DIAGNOSTIC_FEATURES) - set(frame.columns))
    if missing:
        raise AssertionError(f'candidate values missing diagnostic features: {missing}')
    numeric = frame.loc[:, COMBINATION_DIAGNOSTIC_FEATURES].apply(
        pd.to_numeric, errors='raise'
    )
    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise AssertionError('candidate values contain missing/inf')
    train_idx = _validated_partition_indices(
        train_indices, name='internal train indices', row_count=len(numeric)
    )
    internal_val_idx = _validated_partition_indices(
        internal_validation_indices,
        name='internal validation indices',
        row_count=len(numeric),
    )
    if np.intersect1d(train_idx, internal_val_idx).size:
        raise AssertionError('fixed Stage 2 partitions overlap')

    rows = []
    for feature_a, feature_b in COMBINATION_DIAGNOSTIC_PAIRS:
        train_pair = numeric.iloc[train_idx][[feature_a, feature_b]]
        internal_val_pair = numeric.iloc[internal_val_idx][[feature_a, feature_b]]
        train_pearson = float(train_pair.corr(method='pearson').iloc[0, 1])
        internal_val_pearson = float(
            internal_val_pair.corr(method='pearson').iloc[0, 1]
        )
        train_spearman = float(train_pair.corr(method='spearman').iloc[0, 1])
        internal_val_spearman = float(
            internal_val_pair.corr(method='spearman').iloc[0, 1]
        )
        rows.append(
            {
                'feature_a': feature_a,
                'feature_b': feature_b,
                'internal_train_rows': len(train_idx),
                'internal_validation_rows': len(internal_val_idx),
                'train_pearson': train_pearson,
                'internal_val_pearson': internal_val_pearson,
                'train_spearman': train_spearman,
                'internal_val_spearman': internal_val_spearman,
                'pearson_same_direction': _same_correlation_direction(
                    train_pearson, internal_val_pearson
                ),
                'spearman_same_direction': _same_correlation_direction(
                    train_spearman, internal_val_spearman
                ),
                'pearson_abs_difference': _absolute_difference(
                    train_pearson, internal_val_pearson
                ),
                'spearman_abs_difference': _absolute_difference(
                    train_spearman, internal_val_spearman
                ),
            }
        )
    return pd.DataFrame(rows, columns=CANDIDATE_PAIR_REDUNDANCY_COLUMNS)


def build_candidate_pair_complementarity_table(y_true, predictions_by_feature):
    truth = _binary_vector(y_true, name='internal validation labels')
    if tuple(predictions_by_feature) != COMBINATION_DIAGNOSTIC_FEATURES:
        raise AssertionError('prediction keys must match the three diagnostic features in order')
    predictions = {
        feature: _binary_vector(
            predictions_by_feature[feature],
            name=f'{feature} predictions',
            expected_rows=len(truth),
        )
        for feature in COMBINATION_DIAGNOSTIC_FEATURES
    }
    rows = []
    for feature_a, feature_b in COMBINATION_DIAGNOSTIC_PAIRS:
        prediction_a = predictions[feature_a]
        prediction_b = predictions[feature_b]
        a_correct = prediction_a == truth
        b_correct = prediction_b == truth
        both_wrong = (~a_correct) & (~b_correct)
        error_union = (~a_correct) | (~b_correct)
        error_union_count = int(error_union.sum())
        row = {
            'feature_a': feature_a,
            'feature_b': feature_b,
            'internal_validation_rows': len(truth),
            'both_correct': int((a_correct & b_correct).sum()),
            'both_wrong': int(both_wrong.sum()),
            'a_correct_b_wrong': int((a_correct & ~b_correct).sum()),
            'a_wrong_b_correct': int((~a_correct & b_correct).sum()),
            'prediction_disagreement_count': int((prediction_a != prediction_b).sum()),
            'prediction_disagreement_rate': float(np.mean(prediction_a != prediction_b)),
            'error_intersection_count': int(both_wrong.sum()),
            'error_union_count': error_union_count,
            'error_jaccard': (
                float(both_wrong.sum() / error_union_count)
                if error_union_count
                else np.nan
            ),
        }
        for speaker in (0, 1):
            speaker_mask = truth == speaker
            row[f'speaker_{speaker}_a_correct_b_wrong'] = int(
                (speaker_mask & a_correct & ~b_correct).sum()
            )
            row[f'speaker_{speaker}_a_wrong_b_correct'] = int(
                (speaker_mask & ~a_correct & b_correct).sum()
            )
        rows.append(row)
    return pd.DataFrame(rows, columns=CANDIDATE_PAIR_COMPLEMENTARITY_COLUMNS)


def build_baseline_error_change_table(
    y_true,
    baseline_predictions,
    candidate_predictions,
):
    truth = _binary_vector(y_true, name='internal validation labels')
    baseline = _binary_vector(
        baseline_predictions,
        name='MFCC40 baseline predictions',
        expected_rows=len(truth),
    )
    if tuple(candidate_predictions) != COMBINATION_DIAGNOSTIC_FEATURES:
        raise AssertionError('candidate prediction keys do not match diagnostic features')
    baseline_correct = baseline == truth
    rows = []
    for candidate in COMBINATION_DIAGNOSTIC_FEATURES:
        predictions = _binary_vector(
            candidate_predictions[candidate],
            name=f'{candidate} predictions',
            expected_rows=len(truth),
        )
        candidate_correct = predictions == truth
        fixed_errors = int((~baseline_correct & candidate_correct).sum())
        new_errors = int((baseline_correct & ~candidate_correct).sum())
        rows.append(
            {
                'candidate': candidate,
                'internal_validation_rows': len(truth),
                'baseline_wrong_candidate_correct': fixed_errors,
                'baseline_correct_candidate_wrong': new_errors,
                'net_correct_gain': fixed_errors - new_errors,
            }
        )
    return pd.DataFrame(rows, columns=BASELINE_ERROR_CHANGE_COLUMNS)


def fit_training_only_single_feature_predictions(
    aligned_training,
    candidate_frame,
    train_indices,
    internal_validation_indices,
):
    if aligned_training.split != 'Training':
        raise AssertionError('Training-only diagnostic cannot fit from another split')
    train_idx = _validated_partition_indices(
        train_indices, name='internal train indices', row_count=len(aligned_training.y)
    )
    internal_val_idx = _validated_partition_indices(
        internal_validation_indices,
        name='internal validation indices',
        row_count=len(aligned_training.y),
    )
    if np.intersect1d(train_idx, internal_val_idx).size:
        raise AssertionError('fixed Stage 2 partitions overlap')

    diagnostic_baseline = fit_reference_logistic_regression(
        experiment_id='training_only_diagnostic::mfcc40',
        X_train=aligned_training.X_mfcc[train_idx],
        y_train=aligned_training.y[train_idx],
        X_evaluation=aligned_training.X_mfcc[internal_val_idx],
        y_evaluation=aligned_training.y[internal_val_idx],
        feature_names=BASELINE_FEATURE_NAMES,
        training_scope='Training/internal-train',
        evaluation_scope='Training/internal-validation',
    )
    single_results = {}
    for feature in COMBINATION_DIAGNOSTIC_FEATURES:
        candidate_matrix, candidate_feature_names = assemble_experiment_matrix(
            aligned_training, candidate_frame, (feature,)
        )
        if candidate_feature_names != (*BASELINE_FEATURE_NAMES, feature):
            raise AssertionError('diagnostic fit must add exactly one candidate feature')
        assert_single_factor_comparison(
            aligned_training.X_mfcc,
            candidate_matrix,
            aligned_training.y,
            aligned_training.y,
            BASELINE_FEATURE_NAMES,
            candidate_feature_names,
            (feature,),
        )
        single_results[feature] = fit_reference_logistic_regression(
            experiment_id=f'training_only_diagnostic::single::{feature}',
            X_train=candidate_matrix[train_idx],
            y_train=aligned_training.y[train_idx],
            X_evaluation=candidate_matrix[internal_val_idx],
            y_evaluation=aligned_training.y[internal_val_idx],
            feature_names=candidate_feature_names,
            training_scope='Training/internal-train',
            evaluation_scope='Training/internal-validation',
        )
        del candidate_matrix
    return diagnostic_baseline, single_results


In [ ]:
if not RUN_TRAINING_ONLY_COMBINATION_DIAGNOSTICS:
    raise RuntimeError(
        'STOP: Stage 2 fixed split/cache를 확인한 뒤 '
        'RUN_TRAINING_ONLY_COMBINATION_DIAGNOSTICS=True로 바꾸세요.'
    )
if 'stage2_result' not in globals():
    raise RuntimeError(
        'STOP: 기존 Training-only Stage 2 cell의 stage2_result가 필요합니다. '
        'Checkpoint H에서 새 split을 만들지 마세요.'
    )
assert EDA_MANIFEST_PATH.is_file() and EDA_MANIFEST_META_PATH.is_file()
diagnostic_manifest_meta = json.loads(
    EDA_MANIFEST_META_PATH.read_text(encoding='utf-8')
 )
assert diagnostic_manifest_meta['schema_version'] == 'mission2_raw_eda_v1'
assert isinstance(diagnostic_manifest_meta['source_signature'], str)
assert diagnostic_manifest_meta['source_signature']

# Parquet predicate로 Training manifest만 읽는다. official Validation rows는 진단 입력이 아니다.
diagnostic_training_manifest = pd.read_parquet(
    EDA_MANIFEST_PATH,
    filters=[('split', '==', 'Training')],
)
assert len(diagnostic_training_manifest) == 889_358
assert set(diagnostic_training_manifest['split'].astype(str)) == {'Training'}
assert diagnostic_training_manifest['record_key'].is_unique

diagnostic_training_bundle = load_frozen_mfcc_cache(BASELINE_CACHE_ROOT, 'Training')
diagnostic_aligned_training = align_frozen_cache(
    diagnostic_training_bundle,
    diagnostic_training_manifest,
    'Training',
)
del diagnostic_training_bundle
assert diagnostic_aligned_training.X_mfcc.shape == (889_358, 40)

expected_training_cache_features = (
    'zcr',
    'spectral_centroid',
    'spectral_rolloff',
    'spectral_flatness',
    'silence_ratio',
)
if tuple(FULL_TRAINING_CACHE_FEATURES) != expected_training_cache_features:
    raise AssertionError('immutable Training candidate cache feature contract changed')
diagnostic_training_candidate_cache = load_candidate_feature_cache(
    diagnostic_training_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    feature_names=expected_training_cache_features,
    source_signature=diagnostic_manifest_meta['source_signature'],
)

split_audit_frame = stage2_result['split_audit']
if not isinstance(split_audit_frame, pd.DataFrame) or len(split_audit_frame) != 1:
    raise AssertionError('Stage 2 split audit must be a one-row DataFrame')
split_audit = split_audit_frame.iloc[0].to_dict()
diagnostic_train_indices = np.asarray(stage2_result['train_indices'], dtype=np.int64)
diagnostic_internal_validation_indices = np.asarray(
    stage2_result['internal_validation_indices'], dtype=np.int64
)
validate_training_only_combination_diagnostic_scope(
    source_split=stage2_result['scope']['source_split'],
    source_rows=stage2_result['scope']['source_rows'],
    official_validation_used=stage2_result['scope']['official_validation_used'],
    split_audit=split_audit,
    feature_names=COMBINATION_DIAGNOSTIC_FEATURES,
    feature_pairs=COMBINATION_DIAGNOSTIC_PAIRS,
)
assert len(diagnostic_train_indices) == FIXED_STAGE2_SPLIT_CONTRACT['train_rows']
assert len(diagnostic_internal_validation_indices) == FIXED_STAGE2_SPLIT_CONTRACT['internal_validation_rows']
diagnostic_train_calls = set(
    diagnostic_aligned_training.tracking.iloc[diagnostic_train_indices]['call_id'].astype(str)
)
diagnostic_internal_validation_calls = set(
    diagnostic_aligned_training.tracking.iloc[diagnostic_internal_validation_indices]['call_id'].astype(str)
)
assert len(diagnostic_train_calls) == FIXED_STAGE2_SPLIT_CONTRACT['train_calls']
assert len(diagnostic_internal_validation_calls) == FIXED_STAGE2_SPLIT_CONTRACT['internal_validation_calls']
assert diagnostic_train_calls.isdisjoint(diagnostic_internal_validation_calls)

diagnostic_candidate_values = align_extra_feature_matrix(
    diagnostic_aligned_training,
    diagnostic_training_candidate_cache,
    COMBINATION_DIAGNOSTIC_FEATURES,
)
diagnostic_candidate_frame = pd.DataFrame(
    diagnostic_candidate_values,
    columns=COMBINATION_DIAGNOSTIC_FEATURES,
)
candidate_pair_redundancy_table = build_candidate_pair_redundancy_table(
    diagnostic_candidate_frame,
    diagnostic_train_indices,
    diagnostic_internal_validation_indices,
)

diagnostic_baseline_result, diagnostic_single_results = (
    fit_training_only_single_feature_predictions(
        diagnostic_aligned_training,
        diagnostic_training_candidate_cache,
        diagnostic_train_indices,
        diagnostic_internal_validation_indices,
    )
)
diagnostic_y_true = diagnostic_aligned_training.y[
    diagnostic_internal_validation_indices
]
diagnostic_candidate_predictions = {
    feature: diagnostic_single_results[feature].y_pred
    for feature in COMBINATION_DIAGNOSTIC_FEATURES
}
candidate_pair_complementarity_table = (
    build_candidate_pair_complementarity_table(
        diagnostic_y_true,
        diagnostic_candidate_predictions,
    )
)
candidate_baseline_error_change_table = build_baseline_error_change_table(
    diagnostic_y_true,
    diagnostic_baseline_result.y_pred,
    diagnostic_candidate_predictions,
)

display(candidate_pair_redundancy_table)
display(candidate_pair_complementarity_table)
display(candidate_baseline_error_change_table)
print('TRAINING-ONLY DIAGNOSTIC COMPLETE: no combination model was fit and no experiment was selected.')


## 조합 전 진단 해석 — 실행 후 사용자가 작성

- **spectral_rolloff ↔ zcr redundancy:**  
  Pearson 상관계수는 internal train `0.605`, internal validation `0.608`, Spearman은 각각 `0.657`, `0.661`로 나타났다. 두 partition에서 방향과 크기가 거의 동일해 분포는 안정적이다. 세 pair 중에서는 상대적으로 중복성이 가장 낮아, 두 feature가 완전히 같은 정보를 표현한다고 보기는 어렵다.

- **spectral_rolloff ↔ spectral_centroid redundancy:**  
  Pearson은 약 `0.932`, Spearman은 약 `0.954`로 매우 높은 양의 상관관계를 보였다. train과 internal validation에서도 거의 동일했다. 따라서 두 feature는 상당히 유사한 spectral 정보를 공유할 가능성이 높다.

- **zcr ↔ spectral_centroid redundancy:**  
  Pearson은 약 `0.835~0.836`, Spearman은 약 `0.804~0.807`로 높은 양의 상관관계를 보였다. `rolloff ↔ centroid`보다는 낮지만 `rolloff ↔ zcr`보다는 중복성이 크다.

- **rolloff와 zcr의 오류 보완성:**  
  두 모델의 prediction disagreement rate는 `2.533%`, error Jaccard는 `0.872`였다. rolloff가 틀리고 zcr가 맞힌 경우가 `2,182건`, 반대로 rolloff가 맞고 zcr가 틀린 경우가 `2,324건`이었다. speaker=0과 speaker=1 모두에서 양방향 보완 사례가 확인되어, 높은 오류 중복에도 불구하고 서로 다른 오류를 수정하는 여지가 존재한다.

- **rolloff와 centroid의 오류 보완성:**  
  disagreement rate는 `1.353%`, error Jaccard는 `0.929`로 세 pair 중 오류 구조가 가장 유사했다. rolloff가 틀리고 centroid가 맞힌 경우는 `1,130건`, 반대는 `1,277건`이었다. Feature correlation과 error overlap이 모두 매우 높아 첫 조합 후보로는 중복성이 큰 편이다.

- **zcr와 centroid의 오류 보완성:**  
  disagreement rate는 `2.765%`로 세 pair 중 가장 높고 error Jaccard는 `0.861`로 가장 낮았다. zcr가 틀리고 centroid가 맞힌 경우 `2,457건`, 반대는 `2,462건`으로 거의 대칭적이었다. 따라서 prediction 관점의 보완성은 크지만, 두 feature 자체의 Pearson correlation이 약 `0.836`으로 높은 편이라는 점을 함께 고려해야 한다.

- **baseline 대비 각 candidate의 오류 수정/신규 오류:**  
  `spectral_rolloff`는 baseline 오류 `1,869건`을 수정하고 `1,517건`의 신규 오류를 만들어 net gain `+352건`이었다. `zcr`는 `1,334건` 수정 / `1,124건` 신규 오류로 net gain `+210건`, `spectral_centroid`는 `1,608건` 수정 / `1,403건` 신규 오류로 net gain `+205건`이었다. Training-only 기준 단일 candidate 중 spectral_rolloff의 순개선이 가장 컸다.

- **조합 후보를 선택할 근거:**  
  Training-only 진단에서 `spectral_rolloff`가 가장 큰 baseline 대비 net gain을 보였으므로 첫 조합의 기준 feature로 유지한다. 그 파트너로는 `spectral_centroid`보다 `zcr`가 feature correlation과 error overlap이 낮고, rolloff가 틀린 `2,182건`을 zcr가 맞힌 사례도 확인됐다. 따라서 첫 조합 실험 후보는 `spectral_rolloff + zcr`가 가장 타당하다.

- **아직 확인되지 않은 점:**  
  두 feature를 실제로 동시에 입력했을 때 정보가 추가적으로 활용되는지는 아직 확인되지 않았다. 개별 모델의 오류 보완성이 실제 2-feature 모델의 Accuracy 향상으로 이어진다고 보장할 수 없으며, feature interaction, scaler 이후 효과, Recall 변화, high-confidence error 변화도 아직 미확인이다.

- **다음 experiment 결정:**  
  **E05를 `MFCC40 + spectral_rolloff + zcr`의 42차원 조합 실험으로 precommit한다.** 이 결정은 Training-only Checkpoint H의 redundancy, prediction complementarity, baseline error-change 결과를 근거로 한다. 실제 성능 개선 여부는 E05 실행 전에는 확정하지 않는다.

> correlation 하나만으로 조합을 결정하지 않는다.  
> official Validation 결과를 이번 조합 후보 선택 근거로 추가 사용하지 않는다.  
> 실제 조합 experiment는 이 진단을 검토한 뒤 별도로 precommit한다.

# Checkpoint I — E05 사전 확정 조합 실험

E03/E04는 기존 Training-only Stage 2 단일-feature 순위로 사전 확정한 실험이고, E05는 그 두 실험과 별도로 **Checkpoint H의 Training-only 조합 진단을 검토한 뒤** 사전 확정한 첫 조합 실험이다.

- fixed split: Training 889,358행 → internal train 711,467행 / internal validation 177,891행, `seed=42`, `test_size=0.2`, call_id overlap 0
- official Validation 사용: `False`
- baseline error-change: rolloff `+352`, zcr `+210`, centroid `+205` net correct gain
- rolloff ↔ zcr redundancy: Pearson `0.605028 / 0.607525`, Spearman `0.656679 / 0.661497` (train / internal validation)
- rolloff ↔ zcr complementarity: disagreement rate `0.025330`, error Jaccard `0.871869`, directional corrections `2324 / 2182`
- rolloff ↔ centroid는 disagreement rate `0.013531`, error Jaccard `0.929455`로 더 유사한 오류 구조를 보였다.

따라서 **E05=`MFCC40 + spectral_rolloff + zcr`를 E05 official Validation 결과를 보기 전에 고정한다.** 이 기록은 조합 성능 향상, winner 또는 승격을 뜻하지 않는다.

고정 feature 순서는 `frozen MFCC40` 40개 다음에 `spectral_rolloff`, `zcr`이며 총 42차원이다. Validation에서는 이미 존재하는 두 immutable 단일-feature cache를 각각 strict validation한 뒤 `record_key`로만 결합한다. exact 2-feature cache를 만들거나 waveform을 재추출하지 않는다.

향후 v2 재실행은 Training-only Stage 2에서 E05 두 feature 조합을 실제로 평가하고 metric/split/data 근거를 동결한 뒤 Official Validation을 연다. 과거 Checkpoint H 진단은 v2 동결 기록이 아니다.


In [ ]:
# E03/E04 single-feature precommit과 분리된 Checkpoint H 기반 combination precommit이다.
PRECOMMITTED_COMBINATION_EXPERIMENTS: dict[str, tuple[str, ...]] = {
    'E05': ('spectral_rolloff', 'zcr'),
}
# 과거 v1 Checkpoint H 요약이며, v2 조합 Stage 2 결과를 대신하지 않는다.
E05_TRAINING_ONLY_PRECOMMIT = {
    'evidence_checkpoint': 'Checkpoint H',
    'source_split': 'Training',
    'source_rows': 889_358,
    'seed': 42,
    'test_size': 0.2,
    'internal_train_rows': 711_467,
    'internal_validation_rows': 177_891,
    'call_id_overlap': 0,
    'official_validation_used': False,
    'selected_pair': PRECOMMITTED_COMBINATION_EXPERIMENTS['E05'],
}

# 먼저 True로 바꾸고 lock cell을 실행한다. 실제 model 실행 switch는 계속 False다.
LOCK_E05_SELECTION = False
RUN_SELECTED_FEATURE_EXPERIMENT = False
SELECTED_EXPERIMENT_ID = 'E05'
SELECTED_FEATURE_NAMES = PRECOMMITTED_COMBINATION_EXPERIMENTS['E05']
SELECTED_CANDIDATE_NAME = '+'.join(SELECTED_FEATURE_NAMES)

print('Historical E05 choice loaded; no v2 Stage 2 precommit is frozen.')


E05 precommit loaded; selection/model execution remain locked.


In [8]:
expected_e05_features = PRECOMMITTED_COMBINATION_EXPERIMENTS['E05']
if tuple(PRECOMMITTED_COMBINATION_EXPERIMENTS) != ('E05',):
    raise AssertionError('E05 combination precommit mapping changed')
if globals().get('CANDIDATE_SELECTION_LOCKED', False) or globals().get('locked_candidate') is not None:
    raise RuntimeError(
        'STOP: stale locked_candidate 또는 이전 E03/E04 selection state가 남아 있습니다. '
        'fresh kernel에서 Setup/Checkpoint A와 E05 config를 다시 실행하세요.'
    )
if not LOCK_E05_SELECTION:
    raise RuntimeError('STOP: E05 precommit을 확인한 뒤 LOCK_E05_SELECTION=True로 바꾸세요.')
if RUN_SELECTED_FEATURE_EXPERIMENT:
    raise RuntimeError('STOP: selection/cache 검증 전 model 실행 switch를 켜지 마세요.')
if (
    SELECTED_EXPERIMENT_ID != 'E05'
    or SELECTED_CANDIDATE_NAME != '+'.join(expected_e05_features)
    or tuple(SELECTED_FEATURE_NAMES) != expected_e05_features
):
    raise AssertionError('E05 selection must exactly match the precommitted ordered tuple')
# The historical Checkpoint H diagnostic did not fit the two-feature model.
# A future v2 E05 lock needs an actual Training-only Stage 2 combination row.
frozen_precommit = None
CANDIDATE_SELECTION_LOCKED = False
if 'stage2_result' not in globals():
    raise RuntimeError('STOP: run Training-only combination Stage 2 before v2 E05 precommit.')
stage2_config = {
    'mfcc': MFCC_CONFIG,
    'reference_model': REFERENCE_MODEL_CONFIG,
    'candidate_extraction': CANDIDATE_EXTRACTION_CONFIG,
}
frozen_precommit = freeze_stage2_precommit(
    stage2_result,
    experiment_id=SELECTED_EXPERIMENT_ID,
    candidate_name=SELECTED_CANDIDATE_NAME,
    feature_names=SELECTED_FEATURE_NAMES,
    config=stage2_config,
    source_signature=eda_manifest_meta['source_signature'],
)
locked_candidate = lock_candidate_selection(
    stage2_result,
    selection_name=frozen_precommit.candidate_name,
    feature_names=frozen_precommit.feature_names,
)
if locked_candidate.feature_names != expected_e05_features:
    raise AssertionError('locked E05 feature order changed')

# Generic runner의 허용 목록에만 E05를 합친다. E03/E04 원래 mapping은 config에 그대로 보존된다.
PRECOMMITTED_EXPERIMENTS = {
    **PRECOMMITTED_EXPERIMENTS,
    **PRECOMMITTED_COMBINATION_EXPERIMENTS,
}
if tuple(PRECOMMITTED_EXPERIMENTS) != ('E03', 'E04', 'E05'):
    raise AssertionError('precommit execution mapping/order changed')
E05_COMBINATION_SELECTION_LOCKED = True
CANDIDATE_SELECTION_LOCKED = True
print('E05 locked from actual future-v2 Training-only Stage 2 evidence:', frozen_precommit)


E05 LOCKED from Checkpoint H Training-only evidence: CandidateSelection(selection_name='spectral_rolloff+zcr', feature_names=('spectral_rolloff', 'zcr'), evidence_stage='full_or_larger_training_call_id_group_split', selection_fingerprint='a0e2490380e5e4888d7b74251c47c8815dd320ba59ef534ce2c368032a8c0576')


In [9]:
def merge_validated_validation_candidate_caches(
    aligned,
    cache_by_feature,
    feature_names,
    *,
    expected_rows,
):
    names = tuple(feature_names)
    expected_names = PRECOMMITTED_COMBINATION_EXPERIMENTS['E05']
    if names != expected_names or tuple(cache_by_feature) != expected_names:
        raise AssertionError('Validation cache merge must preserve the E05 feature order')
    if aligned.split != 'Validation':
        raise AssertionError('E05 Validation cache merge received a non-Validation split')

    frozen_record_keys = np.asarray(aligned.record_keys).astype(str)
    if len(frozen_record_keys) != expected_rows or len(np.unique(frozen_record_keys)) != expected_rows:
        raise AssertionError('frozen Validation row count/key uniqueness mismatch')
    required_tracking = {'record_key', 'call_id'}
    if not required_tracking.issubset(aligned.tracking.columns):
        raise AssertionError('frozen Validation tracking lacks record_key/call_id')
    frozen_tracking = aligned.tracking[['record_key', 'call_id']].reset_index(drop=True).copy()
    frozen_tracking['record_key'] = frozen_tracking['record_key'].astype(str)
    frozen_tracking['call_id'] = frozen_tracking['call_id'].astype(str)
    np.testing.assert_array_equal(
        frozen_tracking['record_key'].to_numpy(dtype=str),
        frozen_record_keys,
    )
    merged = pd.DataFrame({
        'row_position': np.arange(expected_rows, dtype=np.int64),
        'record_key': frozen_record_keys,
        'frozen_call_id': frozen_tracking['call_id'].to_numpy(dtype=str),
    })

    for feature in names:
        cache = cache_by_feature[feature]
        required_columns = {'record_key', 'call_id', feature}
        if not required_columns.issubset(cache.columns):
            raise AssertionError(f'{feature} cache lacks required columns')
        if len(cache) != expected_rows or not cache['record_key'].is_unique:
            raise AssertionError(f'{feature} cache row count/key uniqueness mismatch')
        cache_view = cache[['record_key', 'call_id', feature]].copy()
        cache_view['record_key'] = cache_view['record_key'].astype(str)
        cache_view['call_id'] = cache_view['call_id'].astype(str)
        cache_view = cache_view.rename(columns={'call_id': f'{feature}_cache_call_id'})
        merged = merged.merge(
            cache_view,
            on='record_key',
            how='outer',
            indicator='_cache_merge',
            validate='one_to_one',
        )
        if len(merged) != expected_rows or not (merged['_cache_merge'] == 'both').all():
            raise AssertionError(f'{feature} cache keys differ from frozen Validation')
        cache_call_id_column = f'{feature}_cache_call_id'
        if not (merged['frozen_call_id'] == merged[cache_call_id_column]).all():
            raise AssertionError(f'{feature} cache call_id alignment mismatch')
        merged = merged.drop(columns=['_cache_merge', cache_call_id_column])

    merged = merged.sort_values('row_position', kind='stable').reset_index(drop=True)
    if len(merged) != expected_rows or not merged['record_key'].is_unique:
        raise AssertionError('merged Validation candidate row count/key uniqueness mismatch')
    np.testing.assert_array_equal(
        merged['record_key'].to_numpy(dtype=str),
        frozen_record_keys,
    )
    candidate_values = merged[list(names)].to_numpy(dtype=float)
    if not np.isfinite(candidate_values).all():
        raise AssertionError('merged E05 Validation features contain missing/inf')
    candidate_frame = merged[['record_key', *names]].copy()
    if tuple(candidate_frame.columns[1:]) != expected_names:
        raise AssertionError('final E05 candidate feature order changed')
    return candidate_frame


expected_e05_features = PRECOMMITTED_COMBINATION_EXPERIMENTS['E05']
active_precommit = require_stage2_precommit(
    globals().get('frozen_precommit'),
    experiment_id=SELECTED_EXPERIMENT_ID,
    candidate_name=SELECTED_CANDIDATE_NAME,
    feature_names=SELECTED_FEATURE_NAMES,
    config=stage2_config,
    source_signature=eda_manifest_meta['source_signature'],
)
expected_e05_features = active_precommit.feature_names
if globals().get('E05_COMBINATION_SELECTION_LOCKED', False) is not True:
    raise RuntimeError('STOP: E05 precommit lock must pass before any Validation cache merge.')
if globals().get('CANDIDATE_SELECTION_LOCKED', False) is not True:
    raise RuntimeError('STOP: generic candidate selection lock is not open for E05.')
if tuple(PRECOMMITTED_COMBINATION_EXPERIMENTS) != ('E05',):
    raise RuntimeError('STOP: E05 combination precommit mapping is missing or changed.')
active_locked_candidate = globals().get('locked_candidate')
if (
    SELECTED_EXPERIMENT_ID != 'E05'
    or tuple(SELECTED_FEATURE_NAMES) != expected_e05_features
    or active_locked_candidate is None
    or active_locked_candidate.selection_name != '+'.join(expected_e05_features)
    or tuple(active_locked_candidate.feature_names) != expected_e05_features
):
    raise RuntimeError('STOP: stale locked_candidate or non-E05 selection state detected.')
if RUN_SELECTED_FEATURE_EXPERIMENT:
    raise RuntimeError('STOP: validate/merge caches before enabling the model experiment.')
if globals().get('BASELINE_GATE_PASSED', False) is not True:
    raise RuntimeError('STOP: exact baseline reproduction gate must pass first.')

# Training은 기존 immutable 5-feature cache를 strict validation 후 그대로 재사용한다.
training_candidate_cache = load_candidate_feature_cache(
    full_eda_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    feature_names=FULL_TRAINING_CACHE_FEATURES,
    source_signature=eda_manifest_meta['source_signature'],
)

# Validation 단일-feature cache를 각각 exact metadata/source/key/finite 계약으로 읽는다.
validation_cache_by_feature = {
    feature: load_candidate_feature_cache(
        full_eda_manifest,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Validation',
        feature_names=(feature,),
        source_signature=eda_manifest_meta['source_signature'],
    )
    for feature in expected_e05_features
}
validation_candidate_cache = merge_validated_validation_candidate_caches(
    aligned_validation,
    validation_cache_by_feature,
    expected_e05_features,
    expected_rows=EXPECTED_UTTERANCE_COUNTS['Validation'],
)
if tuple(validation_candidate_cache.columns[1:]) != expected_e05_features:
    raise AssertionError('final Validation candidate frame changed E05 feature order')

# Generic matrix assembly의 E05 42-dimension/float32/MFCC40 preservation을 fit 전에 확인한다.
e05_validation_matrix_preview, e05_validation_feature_names = assemble_experiment_matrix(
    aligned_validation,
    validation_candidate_cache,
    expected_e05_features,
)
if e05_validation_matrix_preview.shape != (EXPECTED_UTTERANCE_COUNTS['Validation'], 42):
    raise AssertionError('E05 Validation matrix must have shape (111947, 42)')
if e05_validation_matrix_preview.dtype != np.dtype(np.float32):
    raise AssertionError('E05 model matrix must remain float32')
np.testing.assert_array_equal(
    e05_validation_matrix_preview[:, :40],
    np.asarray(aligned_validation.X_mfcc, dtype=np.float32),
)
if e05_validation_feature_names != (*BASELINE_FEATURE_NAMES, *expected_e05_features):
    raise AssertionError('E05 feature names/order changed')
print('E05 caches validated/merged in frozen Validation order; no cache was written.')
print('E05 pre-fit Validation matrix:', e05_validation_matrix_preview.shape, e05_validation_matrix_preview.dtype)


E05 caches validated/merged in frozen Validation order; no cache was written.
E05 pre-fit Validation matrix: (111947, 42) float32


### E05 결과 해석 — 실행 후 사용자가 작성

> 아래 E02/E05 수치와 비교는 과거 v1 결과다. 현재 v2 코드로 재평가하지 않았다.

- Accuracy 및 baseline delta: Official Validation Accuracy는 `0.8131169214`로, E00 baseline `0.8091060948` 대비 `+0.0040108266` (`+0.401%p`) 개선되었다. 정답 수 기준 baseline 대비 `+449건`이다.  
  기존 단일-feature 최고였던 E02 `spectral_rolloff`의 Accuracy `0.8115090177`보다도 `+0.0016079037` (`+0.161%p`), 정답 수 기준 `+180건` 개선되었다. 따라서 `zcr`를 `spectral_rolloff`에 추가했을 때 point estimate 기준 incremental gain이 확인됐다.

- E02 spectral_rolloff 단독 대비 Accuracy 변화: E02 `0.811509` → E05 `0.813117`로 약 `+0.161%p` 개선되었다. Checkpoint H에서 Training-only redundancy/complementarity 진단을 통해 세운 `rolloff + zcr` 조합 가설과 방향이 일치했다.

- speaker=0/1 Recall 변화: speaker=0 Recall은 `0.7743894512`로 baseline 대비 `+0.0094159161` (`+0.942%p`) 개선됐다. E02 대비로도 약 `+0.442%p` 개선됐다.  
  speaker=1 Recall은 `0.8490069189`로 baseline 및 E02 대비 `-0.0009982445` (`-0.100%p`) 하락했다.  
  따라서 E05의 Accuracy 향상은 주로 speaker=0 분류 개선에서 발생했으며, speaker=1에서는 작은 trade-off가 존재한다.

- high-confidence error 변화: High-confidence error는 baseline `2,086건`에서 `2,158건`으로 `+72건` 증가했고, E02 `2,123건`보다도 `+35건` 증가했다. 따라서 Accuracy는 개선됐지만 고확신 오답까지 개선된 것은 아니며, 전체 error quality가 일관되게 좋아졌다고 해석하지 않는다.

- duration/overlap/confidence slice 관찰: E02 대비 very_short Accuracy는 소폭 하락했지만 short, medium, long, very_long slice는 모두 개선됐다.  
  no_overlap Accuracy는 `0.847442`, overlap Accuracy는 `0.768803`으로 E02 대비 각각 약 `+0.160%p`, `+0.162%p` 개선되어 특정 overlap 조건에만 의존한 개선은 관찰되지 않았다.  
  confidence가 높을수록 Accuracy가 증가하는 기존 패턴은 유지됐으나 high-confidence error 총수는 증가했다.

- leakage/evaluation mismatch 재점검: E05는 Checkpoint H의 Training-only redundancy/complementarity/error-change 진단을 근거로 official Validation 결과를 보기 전에 `spectral_rolloff + zcr` 조합으로 precommit했다.  
  E00~E04와 동일한 MFCC40, StandardScaler Training-only fit, LogisticRegression, float32 numerical path 및 official Validation 평가 계약을 유지했다. Validation candidate cache는 strict validation 후 record_key 기준 one-to-one merge했으며 최종 입력은 `(111947, 42)` float32로 확인됐다. 금지 metadata는 model feature에 포함되지 않았다.

- 조합이 단일 feature 대비 추가 signal을 제공했는지: E05는 E02보다 Accuracy가 약 `+0.161%p`, 정답 수 `+180건` 개선되어 `zcr`가 `spectral_rolloff` 단독 위에 추가적인 예측 signal을 제공한 것으로 관찰된다. 다만 이 개선은 주로 speaker=0 Recall 향상에서 발생했고 speaker=1 Recall과 high-confidence error에는 trade-off가 존재한다.

- 과거 v1 당시 결정: LogisticRegression 기반 handcrafted feature 실험 중 E05가 가장 높은 official Validation Accuracy를 기록했으므로 당시 최고 handcrafted LR candidate로 유지했다. 다만 개선 폭의 통계적 유의성은 paired prediction 기반 검정 없이 확정하지 않았다. 이후 모델 구조 비교 계획도 과거 v1 결과에 근거한 결정이며, v2의 현재 성능 결론은 아니다.

> E05 실행 전에는 개선 여부를 작성하지 않는다.  
> E05가 E00보다 좋아도 E02보다 나쁘면 조합 효과를 별도로 평가한다.  
> 큰 성능 상승이면 leakage/evaluation mismatch를 먼저 확인한다.

## Mission 2 Feature Experiment 총정리

> 아래 E02/E05 수치와 결론은 과거 v1 결과다. 현재 v2 코드로 재평가하지 않았다.

- Frozen baseline은 `MFCC40 + StandardScaler + LogisticRegression`이며 Official Validation Accuracy는 `0.809106`이다.
- `duration_ms`는 거의 추가 효과가 없었다.
- 단일 acoustic feature 중 `spectral_rolloff`가 가장 큰 개선을 보였고, `zcr`, `spectral_centroid`도 작은 추가 signal을 보였다.
- Training-only 조합 전 진단에서 `spectral_rolloff + zcr`가 상대적으로 낮은 redundancy와 오류 보완성을 보여 E05 조합으로 precommit했다.
- 과거 v1 E05 `MFCC40 + spectral_rolloff + zcr`는 Accuracy `0.813117`로 당시 실험 중 최고 성능을 기록했다.
  - E00 대비 `+0.401%p`
  - E02 rolloff 단독 대비 `+0.161%p`
  - 정답 수 기준 E02 대비 `+180건`
- E05 개선은 주로 speaker=0 Recall 향상에서 발생했으며, speaker=1 Recall은 소폭 하락했다.
- High-confidence error는 E00/E02보다 증가했으므로 모든 오류 구조가 개선된 것은 아니다.
- 과거 v1 당시 handcrafted Logistic Regression 기준 최종 후보는 `MFCC40 + spectral_rolloff + zcr`로 기록했다.
- 추가 feature 조합을 Official Validation에서 반복 탐색하지 않고 Feature Experiment 단계는 여기서 종료한다.
- 당시 다음 단계 계획은 **42D feature set을 고정하고 모델 구조만 변경하여 Logistic Regression과 비선형 모델을 비교하는 것**이었다. v2 성능 재확인 전에는 이 수치를 현재 성능으로 사용하지 않는다.

> 과거 v1 결론은 point estimate 기준이며, E05가 E02보다 통계적으로 유의하게 우수하다는 판단은 별도 paired comparison 없이 확정하지 않았다.